
<div style="background: linear-gradient(135deg, #1a237e 0%, #00695C 100%); padding: 40px 36px; border-radius: 12px; margin-bottom: 8px;">
<h1 style="color: white; font-size: 1.9em; font-weight: 700; margin: 0 0 10px 0; letter-spacing: -0.5px;">
🏘️ Modelling Housing-Based Financial Vulnerability<br>and Insurance Risk Among Kenyan Households
</h1>
<hr style="border-color: rgba(255,255,255,0.3); margin: 16px 0;">
<p style="color: rgba(255,255,255,0.92); margin: 4px 0; font-size: 1.05em;">
<strong style="color:white;">Dataset:</strong> 2023/24 Kenya Housing Survey (KNBS) · 21,347 households · 47 counties
</p>
<p style="color: rgba(255,255,255,0.92); margin: 4px 0; font-size: 1.05em;">
<strong style="color:white;">Methodology:</strong> CRISP-DM · Five-Dimension HFVS · Gradient Boosting + Deep Learning
</p>
<p style="color: rgba(255,255,255,0.92); margin: 4px 0; font-size: 1.05em;">
<strong style="color:white;">Student:</strong> Valerie Jerono &nbsp;|&nbsp; MSc Data Science &amp; Analytics, Strathmore University / iLabAfrica
</p>
<p style="color: rgba(255,255,255,0.92); margin: 4px 0; font-size: 1.05em;">
<strong style="color:white;">Supervisor:</strong> Dr. Kennedy Senagi &nbsp;|&nbsp; @iLabAfrica Centre
</p>
</div>



## 📋 Notebook Architecture

This is the **single, self-contained dissertation notebook** — one file, one story, run top-to-bottom.
Every cell builds directly on the one before it. Every code block is preceded by *what it does* and
followed by *why the result matters*.

| Phase | Name | Scientific Purpose |
|:---:|---|---|
| **0** | Environment Setup | Reproducible infrastructure — Drive, libraries, paths, constants |
| **1** | Business Understanding | Frame the research problem in actuarial and policy terms |
| **2** | Data Understanding | Profile 21,347 households across 6 core survey files |
| **3** | Data Preparation | Engineer five vulnerability dimensions; build the HFVS composite |
| **4** | Exploratory Data Analysis | Distributions, correlations, compound exposure, affordability gap |
| **5** | Feature Engineering | Leakage-corrected proxy features, interaction terms, spatial CV |
| **6** | Modelling | Logistic Regression, LightGBM, XGBoost, TabNet, MLP |
| **7** | Evaluation and Interpretability | AUC, calibration, SHAP, ROC/PR curves |
| **8** | County Risk Mapping | Spatial aggregation, IRA validation, AHP alignment, urban-rural gap |
| **9** | Conclusions | Synthesis of findings for academic and policy audience |
| **10** | Recommendations | Stakeholder actions and research next steps |

> **How to read this notebook:** Each section opens with a markdown cell explaining the *scientific
> rationale* before any code runs. Read these first — they tell the story. The code then
> *executes* the story, and the output *confirms* it.

---



---
# ⚙️ Phase 0 — Environment Setup

This notebook is designed to run on Google Colab Pro+ with a T4 GPU. Before any analysis can happen, we need a reproducible infrastructure: the same libraries installed, the same file paths defined, and the same random seed set everywhere. Without this foundation, results cannot be compared between runs or between collaborators.

Google Drive is used as the persistent file system rather than Colab's ephemeral local storage. When a Colab session restarts, everything in `/content` is lost. By writing all parquet files, models, and figures to a mounted Drive folder, we ensure that intermediate outputs survive session interruptions. For a dissertation pipeline that takes 45-60 minutes to run end-to-end, this is not a convenience — it is a scientific necessity.

Polars is chosen over Pandas as the primary data loader because the KHS dataset spans 21,347 households across 392 columns, and Polars completes groupby and join operations roughly five times faster than Pandas on this scale. After initial loading and joining, the working dataframe is converted to Pandas for compatibility with scikit-learn and the modelling ecosystem.

The fixed random seed (`SEED = 42`) is propagated to NumPy, scikit-learn, XGBoost, LightGBM, PyTorch, and Python's own `random` module. Without this, any stochastic operation — from train-test splitting to neural network weight initialisation — will produce different results on different runs, making the dissertation unreproducible. The colour palette is defined here once and inherited by every chart in the notebook. The colours are inspired loosely by Kenya's flag: deep green (TEAL), red, and black-adjacent dark tones, alongside academic blues and ambers used to indicate moderate and high vulnerability states.


In [ ]:
# ── 0.1  Mount Google Drive and clone/pull the repository ─────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
os.chdir('/content')
!git clone https://github.com/VAL-Jerono/KHS_housing_dissertation.git 2>/dev/null || \
    (cd KHS_housing_dissertation && git pull)
os.chdir('KHS_housing_dissertation')
sys.path.insert(0, 'src')
print("✓ Drive mounted. Repository ready.")


In [ ]:
# ── 0.2  Install all required dependencies (first run only — ~90 seconds) ──────
!pip install -q polars pyarrow scikit-learn matplotlib seaborn scipy \
    xgboost lightgbm shap pytorch-tabnet geopandas statsmodels \
    mapclassify contextily imbalanced-learn joblib upsetplot
print("✓ All packages installed.")


In [ ]:
# ── 0.3  Core imports ─────────────────────────────────────────────────────────
import json, warnings, pickle
import numpy as np
import pandas as pd
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.gridspec as gridspec
import matplotlib.colors as mcolors
from matplotlib.patches import Patch
import seaborn as sns
from pathlib import Path
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

# ── Sklearn ────────────────────────────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import (StratifiedKFold, StratifiedGroupKFold,
                                      KFold, cross_val_predict,
                                      RandomizedSearchCV, train_test_split)
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                              classification_report, confusion_matrix,
                              mean_squared_error, r2_score, mean_absolute_error,
                              precision_recall_curve, roc_curve)
from sklearn.preprocessing import RobustScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif
from sklearn.calibration import calibration_curve
from sklearn.impute import SimpleImputer

# ── Boosting + Deep Learning ───────────────────────────────────────────────────
import xgboost as xgb
import lightgbm as lgb
import shap
import torch
import torch.nn as nn
from pytorch_tabnet.tab_model import TabNetRegressor, TabNetClassifier

import joblib
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
print(f"✓ All imports loaded.")
print(f"  XGBoost {xgb.__version__} | LightGBM {lgb.__version__} | SHAP {shap.__version__}")
print(f"  PyTorch {torch.__version__}")


In [ ]:
# ── 0.4  Paths, constants, colour palette, plot style, county map ─────────────

DRIVE  = Path('/content/drive/MyDrive/KHS_Dissertation')
PQ     = DRIVE / 'data' / 'parquet'
RAW    = DRIVE / 'data' / 'raw'
OUT    = DRIVE / 'outputs'
FIGS   = OUT / 'figures'
TABS   = OUT / 'tables'
MODS   = OUT / 'models'
SHPS   = DRIVE / 'data' / 'shapefiles'
for p in [FIGS, TABS, MODS, SHPS]: p.mkdir(parents=True, exist_ok=True)

# ── Global constants ───────────────────────────────────────────────────────────
N_FOLDS        = 5       # Cross-validation folds
SEED           = 42      # Random seed — propagated everywhere
HFVS_THRESHOLD = 0.60    # Score above which a household is classified "high vulnerability"

# ── Colour palette (defined once, used in every chart) ─────────────────────────
TEAL   = '#00695C'
RED    = '#B71C1C'
AMBER  = '#E65100'
BLUE   = '#1565C0'
PURPLE = '#6A1B9A'
GRAY   = '#546E7A'
DARK   = '#2C2C2A'

# ── Publication plot style ─────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'        : 140,
    'figure.facecolor'  : 'white',
    'axes.facecolor'    : '#F8F8F6',
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.titlesize'    : 13,
    'axes.titleweight'  : '600',
    'axes.labelsize'    : 11,
    'xtick.labelsize'   : 9,
    'ytick.labelsize'   : 9,
    'font.family'       : 'sans-serif',
    'legend.framealpha' : 0.9,
    'legend.fontsize'   : 9,
})

# ── All 47 Kenya counties (survey integer code → county name string) ─────────
COUNTY_MAP = {
     1:'Mombasa',        2:'Kwale',          3:'Kilifi',         4:'Tana River',
     5:'Lamu',           6:'Taita-Taveta',   7:'Garissa',        8:'Wajir',
     9:'Mandera',       10:'Marsabit',      11:'Isiolo',        12:'Meru',
    13:'Tharaka-Nithi', 14:'Embu',          15:'Kitui',         16:'Machakos',
    17:'Makueni',       18:'Nyandarua',     19:'Nyeri',         20:'Kirinyaga',
    21:"Murang'a",      22:'Kiambu',        23:'Turkana',       24:'West Pokot',
    25:'Samburu',       26:'Trans Nzoia',   27:'Uasin Gishu',   28:'Elgeyo-Marakwet',
    29:'Nandi',         30:'Baringo',       31:'Laikipia',      32:'Nakuru',
    33:'Narok',         34:'Kajiado',       35:'Kericho',       36:'Bomet',
    37:'Kakamega',      38:'Vihiga',        39:'Bungoma',       40:'Busia',
    41:'Siaya',         42:'Kisumu',        43:'Homa Bay',      44:'Migori',
    45:'Kisii',         46:'Nyamira',       47:'Nairobi',
}

print(f"✓ Environment configured.")
print(f"  FIGS: {FIGS}")
print(f"  TABS: {TABS}")
print(f"  Counties defined: {len(COUNTY_MAP)}")



---
# 🏢 Phase 1 — Business Understanding

## 1.1 The Housing Crisis Context

Kenya faces a housing deficit of approximately two million units, with an annual shortfall of around 200,000 new dwellings against a population growing at one million per year. This deficit is not evenly distributed. It is concentrated in informal settlements — the peri-urban and urban fringe areas where land tenure is informal, structures are non-permanent, and basic services are absent. In these settlements, the gap between housing need and housing provision has widened for two decades, even as macroeconomic growth indicators have improved.

The April 2024 floods placed this structural deficit in stark relief. Above-average rainfall caused 270 deaths and displaced more than 200,000 people across the country. The households most severely affected were those already living in flood-adjacent informal settlements without formal tenure documents, without savings to absorb losses, and without insurance to recover from them. The government's subsequent eviction orders in high-risk riparian zones displaced an additional cohort of residents who had no legal standing to resist. What the media called a natural disaster was, in material terms, the foreseeable consequence of a housing market that had systematically excluded its most vulnerable participants from safety, security, and finance simultaneously.

Kenya's household insurance penetration stands at approximately 2.3 percent. This is not because Kenyan households lack demand for risk protection. Survey evidence and qualitative research consistently show high willingness to pay for security products. The gap exists because insurers lack the granular data to price risk at the household or community level. Without pricing data, products cannot be designed. Without products, demand cannot be revealed. This is a measurement vacuum, not a market failure in the conventional sense.

Policymakers attempting to target housing interventions have historically relied on income quintiles and geographic overlays derived from census data. These instruments miss the multidimensional nature of housing vulnerability entirely. A household in the third income quintile may be simultaneously in a flood zone, renting without a written lease, and spending forty percent of expenditure on rent — a risk profile that no income quintile captures. The HFVS framework proposed in this dissertation is designed to fill precisely this measurement gap.

## 1.2 Research Question

> Can a machine learning model trained on nationally representative household microdata produce a reliable, granular Housing Financial Vulnerability Score (HFVS) that serves as an actuarially valid risk variable for insurance pricing and policy targeting across all 47 Kenyan counties?

This question is simultaneously an academic contribution and a practical instrument. Academically, it tests whether composite vulnerability indices derived from direct survey measurement can be approximated by proxy models robust enough for field deployment — a methodological question with relevance across East Africa. Practically, it produces a county-level risk map that the Insurance Regulatory Authority, individual underwriters, and the State Department of Housing can use immediately. The IRA can incorporate county HFVS ranks into regulatory guidance for microinsurance pricing. Underwriters can use the framework to design parametric products. The State Department can use the missed-county analysis to recalibrate Affordable Housing Programme site selection away from population-size criteria toward vulnerability-score criteria.

## 1.3 The HFVS Framework

The Housing Financial Vulnerability Score is a five-dimension composite index. Each dimension is constructed from direct survey measurements, normalised to the unit interval, and combined as an equal-weighted mean:

$$\text{HFVS}_i = \frac{D_1 + D_2 + D_3 + D_4 + D_5}{5}$$

Where the five dimensions are:

| Dimension | Core Variables | Actuarial Rationale |
|---|---|---|
| D1 — Financial Stress | Rent-to-expenditure ratio, savings rate, income quintile | Predicts inability to pay premium or absorb uninsured loss |
| D2 — Tenure Insecurity | Land ownership, written lease, eviction threat | Predicts claim eligibility disputes and moral hazard |
| D3 — Physical Hazard | Enumerator-observed flood and mudslide zones, proximity risks | Predicts claim frequency and severity |
| D4 — Dwelling Quality | Floor/wall/roof material durability, overcrowding, floor area | Predicts structural replacement cost and total loss probability |
| D5 — Utility Deprivation | Electricity access, water quality, sanitation, cooking fuel | Predicts compound health claims and indirect losses |

Equal weighting is chosen for interpretability, robustness, and stakeholder legitimacy. Empirically derived weights would optimise for in-sample fit at the cost of stability and explainability.


In [ ]:
# ── 1.4  Visualise the HFVS Framework ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 10); ax.set_ylim(0, 6); ax.axis('off')

dims = [
    ('D1\nFinancial\nStress',    0.8,  RED),
    ('D2\nTenure\nInsecurity',   2.4,  AMBER),
    ('D3\nPhysical\nHazard',     4.0,  PURPLE),
    ('D4\nDwelling\nQuality',    5.6,  BLUE),
    ('D5\nUtility\nDeprivation', 7.2,  GRAY),
]

for label, x, col in dims:
    ax.add_patch(plt.FancyBboxPatch((x, 2), 1.2, 2,
                 boxstyle='round,pad=0.1', fc=col, alpha=0.85, ec='white', lw=2))
    ax.text(x + 0.6, 3.0, label, ha='center', va='center',
            fontsize=8.5, fontweight='600', color='white')
    ax.annotate('', xy=(8.7, 3.0), xytext=(x + 1.2, 3.0),
                arrowprops=dict(arrowstyle='->', color=DARK, lw=1.2, alpha=0.5))

ax.add_patch(plt.FancyBboxPatch((8.7, 2.3), 1.1, 1.4,
             boxstyle='round,pad=0.1', fc=DARK, ec='none'))
ax.text(9.25, 3.0, 'HFVS', ha='center', va='center',
        fontsize=10, fontweight='700', color='white')
ax.text(5.0, 5.2, 'HFVS = (D1 + D2 + D3 + D4 + D5) / 5     [equal weights]',
        ha='center', va='center', fontsize=10, fontweight='600', color=DARK,
        bbox=dict(boxstyle='round', fc='#F0F4F8', ec=GRAY, pad=0.4))
ax.set_title('Housing Financial Vulnerability Score — Five-Dimension Framework',
             fontsize=12, fontweight='700', pad=12)
plt.tight_layout()
plt.savefig(FIGS / 'phase1_hfvs_framework.png', dpi=150, bbox_inches='tight')
plt.show()



## 1.4 Stakeholder Map

| Stakeholder | Decision Need | Success Criterion |
|---|---|---|
| Insurance Regulatory Authority (IRA) | A risk variable that justifies county-differentiated microinsurance pricing | HFVS correlated with IRA loss ratios at Spearman rho > 0.50 |
| Underwriters | Actuarially valid household-level scores for product design | Calibrated probability outputs usable as premium-loading inputs |
| NGOs / UN-Habitat | A poverty-complementary targeting instrument for housing interventions | HFVS identifies households missed by income-quintile filters |
| State Department of Housing | Evidence base for Affordable Housing Programme site selection | County HFVS rank correlated with programme allocation gap |
| Academic reviewers | Methodological rigour, reproducibility, leakage-free modelling | Full open-source pipeline; spatial CV; honest AUC reporting |

## 1.5 CRISP-DM as the Guiding Framework

The Cross-Industry Standard Process for Data Mining (CRISP-DM) was selected as the methodological framework for three reasons. First, it is iterative: the framework mandates returning to earlier phases when modelling results are implausible. Second, it is recognised by industry stakeholders in insurance and financial services, which strengthens the translation of academic findings into practice. Third, it is reproducible: each phase produces documented outputs that the next phase consumes, creating a full audit trail.

Two bugs in the v1 pipeline were caught precisely because CRISP-DM mandates returning to Data Understanding when evaluation results appear suspicious. When the first modelling run produced AUC scores above 0.99, the CRISP-DM protocol required a return to the feature construction phase — where the leakage of formula-ancestor variables into the model was discovered and corrected. A purely linear pipeline would have submitted the 0.99 AUC result without question.

## 1.6 The Four Policy Questions This Study Answers

This dissertation pursues four distinct policy lenses, each of which emerges from the data and runs through all subsequent CRISP-DM phases as a recurring thread.

The first is the hazard-tenure-finance intersection: which households are simultaneously in a flood zone, without tenure security, and financially stressed? This is the triple-exposure profile that defined the 2024 flood victims, and it can now be quantified from the 2023/24 KHS for all 47 counties.

The second is the affordability gap: what is the distance between what households say they can afford (column k25) and what they actually pay (column k05)? This revealed constraint tells us whether households are voluntarily underspending on housing or involuntarily trapped in dwellings they cannot sustain — a distinction with direct implications for the Boma Yangu programme's unit pricing.

The third is the geography of financial exclusion: where does formal housing finance exist and where is it entirely absent? Mortgage and loan file analysis reveals the counties where the Kenya Mortgage Refinance Company and related instruments have zero penetration — precisely the counties where vulnerability is highest.

The fourth is programme alignment: is the Affordable Housing Programme reaching the most vulnerable counties? A Mann-Whitney test on AHP-active versus AHP-inactive counties answers this directly, and the missed-county table provides an immediately actionable shortlist for the next programme tranche.

These four lenses are not separate analyses. They are woven through every phase, culminating in the Phase 9 synthesis and Phase 10 policy table.



---
# 📊 Phase 2 — Data Understanding

Data Understanding is not a formality. In a survey dataset of 21,347 households, 392 columns, and 11 linked files, an analyst who skips this phase will build features on top of miscoded variables, miss structural missingness patterns, and draw conclusions from artifacts of survey design rather than from genuine household characteristics. This phase is where we learn the shape, limits, and inner logic of the data before touching a single variable for analysis.

Five questions guide this phase. First: what is in each file, how many records, and what is the relational key? Second: where are the missing values, and are they random or structural? Third: how are households distributed geographically, and does the sample represent all 47 counties adequately? Fourth: what does the individual-level demographic file tell us about household composition? Fifth: do the mortgage and loan files reveal the geography of financial exclusion? Answering all five before Phase 3 is not optional — it is how science prevents measurement error from entering the analysis pipeline.

The Kenya Housing Survey 2023/24 is stratified by county and urban/rural stratum. Nairobi is oversampled relative to its population share because the survey needed precise urban estimates for the capital. This means that any county-level analysis must use survey weights (the `hhweight` column) to produce nationally representative estimates. Unweighted county means would overstate urban vulnerability and understate rural deprivation in ASAL regions.


In [ ]:
# ── 2.1  Load codebook labels ─────────────────────────────────────────────────
# The KHS uses Stata-style integer codes for categorical variables.
# The JSON label files map code → label string for audit and interpretation.

def load_json_safe(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return {}

HH_VAR  = load_json_safe(PQ / 'household_variable_labels.json')
HH_VAL  = load_json_safe(PQ / 'household_value_labels.json')
DW_VAR  = load_json_safe(PQ / 'dwelling_variable_labels.json')
IND_VAR = load_json_safe(PQ / 'individual_variable_labels.json')

def decode(col, code, source='HH'):
    '''Return human-readable label for a survey code.'''
    store = {'HH': HH_VAL, 'DW': DW_VAR, 'IND': IND_VAR}.get(source, HH_VAL)
    labels = store.get(col.upper(), store.get(col, {}))
    return labels.get(str(int(code)), labels.get(int(code), f'code_{int(code)}'))

def label_lookup(labels, code):
    '''Look up a code in a labels dict; return 'UNKNOWN' if absent.'''
    if pd.isna(code):
        return 'missing'
    return labels.get(str(int(code)), labels.get(int(code), 'UNKNOWN'))

print("✓ Codebook utilities loaded.")
print(f"  HH variable labels : {len(HH_VAR)} columns")
print(f"  HH value labels    : {len(HH_VAL)} coded columns")


In [ ]:
# ── 2.2  File inventory — load all six core files ─────────────────────────────
# Systematic loading ensures we document exactly what data is available
# before any engineering decisions are made.

def load_pq(name):
    path = PQ / f'{name}.parquet'
    if not path.exists():
        print(f"  WARNING: {path} not found — returning empty DataFrame")
        return pd.DataFrame()
    return pl.read_parquet(path).to_pandas()

hh       = load_pq('household')
ind      = load_pq('individual')
dw       = load_pq('dwelling')
county   = load_pq('county')
mortgage = load_pq('mortgage')
loan     = load_pq('loan')

dfs = {'household': hh, 'individual': ind, 'dwelling': dw,
       'county': county, 'mortgage': mortgage, 'loan': loan}

descriptions = {
    'household': 'Primary HH-level file — expenditure, tenure, household roster',
    'individual': 'Individual members — demographics, education, work-age status',
    'dwelling'  : 'Dwelling units — materials, utilities, area, hazard observations',
    'county'    : '47-county reference file — county codes and names',
    'mortgage'  : 'Households with formal mortgage — loan amount, lender, status',
    'loan'      : 'Households with non-mortgage loans — informal and formal lending',
}

print(f"{'File':<14} {'Rows':>8} {'Cols':>6}  Description")
print("─" * 75)
for name, df in dfs.items():
    print(f"  {name:<12} {len(df):>8,} {df.shape[1]:>6}  {descriptions.get(name,'')}")



The KHS data model has a clear relational spine: the household file is the primary grain, with `interview__key` as the join key. The individual file is one-to-many (multiple members per household); it must be aggregated to household grain before joining. The dwelling file records one or more dwelling units per household; we keep only the primary unit. The county file is a 47-row reference table. The mortgage and loan files are sparse — most households have no record in either — and their absence is itself an informative signal about financial exclusion.


In [ ]:
# ── 2.3  Household null audit — classify 392 columns by missingness tier ──────
null_pct = (hh.isnull().mean() * 100).sort_values(ascending=False)

tiers = {
    'Complete  (0%)     ': null_pct == 0,
    'Low       (1-20%)  ': (null_pct > 0)  & (null_pct <= 20),
    'Moderate  (21-60%) ': (null_pct > 20) & (null_pct <= 60),
    'High      (61-90%) ': (null_pct > 60) & (null_pct <= 90),
    'Extreme   (>90%)   ': null_pct > 90,
}

print("Household file — Null audit (all columns):")
print("─" * 60)
for tier, mask in tiers.items():
    count = mask.sum()
    bar   = '█' * (count // 4)
    note  = '<- structural (renter/owner split)' if 'Moderate' in tier else ''
    print(f"  {tier}  {count:>3} cols  {bar} {note}")

fig, ax = plt.subplots(figsize=(11, 5.5))
top30    = null_pct.head(30)
bar_cols = [RED if v > 60 else AMBER if v > 20 else TEAL for v in top30.values]
ax.barh(top30.index[::-1], top30.values[::-1], color=bar_cols[::-1], height=0.65)
ax.axvline(60, color=RED,   lw=1.5, ls='--', alpha=0.7, label='60% threshold — drop')
ax.axvline(20, color=AMBER, lw=1.5, ls='--', alpha=0.7, label='20% threshold — caution')
ax.set_xlabel('% Missing')
ax.set_title('Top 30 Columns by Missingness — KHS Household File (2023/24)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(FIGS / 'phase2_null_audit.png', dpi=150, bbox_inches='tight')
plt.show()



The missingness pattern here is not random and not a data quality failure. The moderate-missingness tier (21-60%) captures columns like `k05` (monthly rent paid) and `k25` (willing-to-spend amount), which are structurally absent for owner-occupiers because the survey only asked renters these questions. Similarly, mortgage-amount columns are missing for the 95-plus percent of households with no formal mortgage. Any imputation strategy that ignores this structure risks introducing systematic bias. For formula variables, structural missingness is handled at the dimension level with group-median fills; for model features, it is handled by the SimpleImputer inside each fold's pipeline.


In [ ]:
# ── 2.4  Geographic distribution — county and residence breakdown ──────────────
hh['county_name'] = hh['a01'].map(COUNTY_MAP)
hh['residence']   = hh['a07_1'].map({1: 'Rural', 2: 'Urban'})

stratum_counts = (
    hh.groupby(['county_name', 'residence'])
      .size().unstack(fill_value=0)
      .assign(total=lambda x: x.sum(axis=1))
      .sort_values('total', ascending=False)
)

rural_total = hh['residence'].eq('Rural').sum()
urban_total = hh['residence'].eq('Urban').sum()

print("Geographic distribution summary:")
print(f"  Rural households : {rural_total:,} ({rural_total/len(hh)*100:.1f}%)")
print(f"  Urban households : {urban_total:,} ({urban_total/len(hh)*100:.1f}%)")
print(f"  Counties covered : {hh['a01'].nunique()} / 47")
print(f"  Median per county: {stratum_counts['total'].median():.0f} households")
print(f"  Range            : {stratum_counts['total'].min()} to {stratum_counts['total'].max()}")
print("\nTop 10 counties by sample size:")
print(stratum_counts[['Rural','Urban','total']].head(10).to_string())


In [ ]:
# ── 2.5  Individual file demographic profile — age, gender, education ──────────
ind['age_n']     = pd.to_numeric(ind['age_cur'], errors='coerce')
ind['gender']    = pd.to_numeric(ind['b04'],     errors='coerce').map({1:'Male', 2:'Female'})
ind['edu_isced'] = pd.to_numeric(ind.get('ken_edu_isced11',
                                          pd.Series(np.nan, index=ind.index)),
                                  errors='coerce')

print("Individual file demographic profile:")
print(f"  Total individuals     : {len(ind):,}")
print(f"  Mean age              : {ind['age_n'].mean():.1f} years")
print(f"  % Female              : {(ind['gender']=='Female').mean()*100:.1f}%")
print(f"  % Under 15 (children) : {(ind['age_n'] < 15).mean()*100:.1f}%")
print(f"  % 65+ (elderly)       : {(ind['age_n'] >= 65).mean()*100:.1f}%")
print(f"  Mean ISCED level      : {ind['edu_isced'].mean():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ind['age_n'].dropna(), bins=40, color=TEAL, alpha=0.85, edgecolor='white')
axes[0].set_xlabel('Age (years)'); axes[0].set_ylabel('Count')
axes[0].set_title('Age Distribution — All Individuals')

isced_counts = ind['edu_isced'].value_counts().sort_index()
axes[1].bar(isced_counts.index.astype(str), isced_counts.values, color=BLUE, alpha=0.85)
axes[1].set_xlabel('ISCED Level'); axes[1].set_ylabel('Count')
axes[1].set_title('Education Level (ISCED) — All Individuals')
plt.tight_layout()
plt.savefig(FIGS / 'phase2_demographics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 2.6  Cross-file join audit — validate key coverage before engineering ──────
hh_keys  = set(hh['interview__key'].dropna())
ind_keys = set(ind['interview__key'].dropna())
dw_keys  = set(dw['interview__key'].dropna())
mtg_keys = set(mortgage['interview__key'].dropna()) if len(mortgage) > 0 else set()
ln_keys  = set(loan['interview__key'].dropna()) if len(loan) > 0 else set()

print("Cross-file join audit:")
print(f"  Household keys             : {len(hh_keys):,}")
print(f"  Individual keys            : {len(ind_keys):,} "
      f"({len(ind_keys & hh_keys)/len(hh_keys)*100:.1f}% covered)")
print(f"  Dwelling keys              : {len(dw_keys):,} "
      f"({len(dw_keys & hh_keys)/len(hh_keys)*100:.1f}% covered)")
print(f"  Mortgage keys              : {len(mtg_keys):,} "
      f"({len(mtg_keys)/len(hh_keys)*100:.2f}% of HHs)")
print(f"  Loan keys                  : {len(ln_keys):,} "
      f"({len(ln_keys)/len(hh_keys)*100:.2f}% of HHs)")

orphan_ind = len(ind_keys - hh_keys)
orphan_dw  = len(dw_keys - hh_keys)
print(f"\n  Orphan individual records  : {orphan_ind}")
print(f"  Orphan dwelling records    : {orphan_dw}")



The join audit confirms the relational integrity of the KHS data. Individual and dwelling files have high household coverage, with any orphan records likely corresponding to survey attrition where the dwelling interview was completed but the household roster was not. These orphan records are excluded during spine construction in Phase 3. The mortgage and loan files, as expected, cover a small fraction of total households — the penetration rates here set the baseline for the finance-exclusion analysis.


In [ ]:
# ── 2.7  Finance file deep profile — mortgage and loan penetration ─────────────
# First: print actual column names to avoid assumptions about column naming
print("MORTGAGE FILE columns:", list(mortgage.columns)[:20] if len(mortgage) > 0 else "File empty")
print("LOAN FILE columns:",     list(loan.columns)[:20]     if len(loan) > 0     else "File empty")

# Penetration rates
n_mortgage_hh  = mortgage['interview__key'].nunique() if len(mortgage) > 0 else 0
n_loan_hh      = loan['interview__key'].nunique()     if len(loan) > 0     else 0
mortgage_pen   = n_mortgage_hh / len(hh)
loan_pen       = n_loan_hh / len(hh)

print(f"\nMortgage penetration : {n_mortgage_hh:,} households ({mortgage_pen*100:.2f}%)")
print(f"Loan penetration     : {n_loan_hh:,} households ({loan_pen*100:.2f}%)")

# Add finance flags to household file
hh['has_mortgage'] = hh['interview__key'].isin(mtg_keys).astype(float)
hh['has_loan']     = hh['interview__key'].isin(ln_keys).astype(float)

# County-level mortgage penetration
county_mtg = (
    hh.groupby(['a01', 'county_name'])['has_mortgage']
      .mean().reset_index()
      .rename(columns={'has_mortgage': 'mortgage_penetration'})
      .sort_values('mortgage_penetration', ascending=True)
)

nat_mean_pen = county_mtg['mortgage_penetration'].mean()
bar_cols_mtg = [RED if v < nat_mean_pen else TEAL for v in county_mtg['mortgage_penetration']]

fig, ax = plt.subplots(figsize=(9, 12))
ax.barh(county_mtg['county_name'], county_mtg['mortgage_penetration'] * 100,
        color=bar_cols_mtg, edgecolor='white', height=0.7)
ax.axvline(nat_mean_pen * 100, color=DARK, lw=1.5, ls='--',
           label=f'National mean ({nat_mean_pen*100:.2f}%)')
legend_elements = [Patch(color=TEAL, label='Above national mean'),
                   Patch(color=RED,  label='Below national mean')]
ax.legend(handles=legend_elements, loc='lower right')
ax.set_xlabel('Mortgage Penetration (%)')
ax.set_title('Formal Mortgage Penetration — All 47 Counties\n(% of households with active mortgage)',
             fontsize=12, fontweight='600')
plt.tight_layout()
plt.savefig(FIGS / 'phase2_mortgage_penetration.png', dpi=180, bbox_inches='tight')
plt.show()

county_mtg.to_csv(TABS / 'county_mortgage_penetration.csv', index=False)



The mortgage penetration map reveals a stark geography of financial exclusion. The counties with the lowest mortgage penetration rates are predominantly ASAL regions — arid and semi-arid lands in the north and northeast that also carry some of the highest physical hazard scores. This pattern matters for two Kenya-specific initiatives: the Kenya Mortgage Refinance Company (KMRC), which provides concessional long-term funding to primary lenders, and the Kenya Mortgage Guarantee Trust (KMGT), which backstops mortgage origination in underserved areas. Both instruments are currently deployed disproportionately toward the counties already above the national penetration mean. The counties that most need them, identified clearly here, have near-zero formal mortgage presence. The policy implication is direct: KMRC's next tranche of refinancing capacity should be explicitly conditional on origination in the bottom-quartile penetration counties identified in this analysis.



## 2.8 Risk Dimension Feasibility

Before building the five HFVS dimensions in Phase 3, it is worth mapping each dimension to its source columns and their known limitations.

**D1 Financial Stress:** Core variables are `c14_1` (total household expenditure), `k05` (monthly rent paid), `c14_2` (savings), and `k27` (accessible loan amount). Missingness in `k05` is structural: it is only asked of renters, with `l15` (estimated rental value) serving as an owner-occupier proxy. The v1 bug in this dimension misidentified the loan-access column, resulting in near-zero variance on the no-loan-access indicator. The corrected v2 uses `k27 == 0` as the flag.

**D2 Tenure Insecurity:** Core variables are `i00` (land ownership status), `k02` (written lease agreement), and `k35` (eviction threat). Two successive bugs were found and corrected: v1 tested `i00 == 2` (a code that does not exist in the KHS), and an interim attempt tested `k02 == 0` (which maps to a different category). The confirmed fix uses `i00 == 0` for no land ownership and `k02 == 2` for no written agreement, as verified from the Stata value label dictionaries.

**D3 Physical Hazard:** Core variables are `e06` (flood zone severity) and `e07` (mudslide zone severity), both enumerator-observed — the highest quality data type in the survey. Proximity sub-indicators (`e08__1` through `e08__6`) capture nearness to swamp, dumpsite, factory, road, river, and quarry. Not all KHS extracts include these sub-columns; the code checks availability and re-normalises weights dynamically.

**D4 Dwelling Quality:** Core variables are `d14` (floor material), `d15` (wall material), and `d16` (roof material). The key methodological point is the inversion: non-durable materials score high on D4 vulnerability. Material codes are verified against the Stata data dictionary and the WHO/UN-Habitat shelter typology.

**D5 Utility Deprivation:** Core variables are `c08` (electricity access), `c01_1` (water source), `c04` (toilet type), `c05` (shared toilet), and `c11` (cooking fuel). Coding is aligned to the WHO/UNICEF JMP 2023 service ladders. A code audit for `c08` and `c11` is printed during dimension construction to provide a verifiable audit trail.



---
# 🔧 Phase 3 — Data Preparation

Data preparation is where the scientific claims of this dissertation are built, one variable at a time. Every choice made here — how to define rent burden, which material codes count as non-durable, what counts as an eviction threat — is a measurement decision with actuarial consequences. This phase documents every decision with the source column, the coding rationale, and the audit output that verifies the implementation is correct.

The architecture of Phase 3 follows the HFVS framework exactly: master spine construction, then each of the five dimensions in sequence, then the composite, then the affordability gap analysis, then the compound exposure flags. Variables that are not HFVS formula ingredients — demographic proxies, interaction terms, and affordability gap descriptors — are clearly separated from dimension components to prevent any possibility of modelling leakage in Phase 5.

The modelling leakage problem deserves emphasis because it derailed the v1 pipeline entirely. When the five dimension scores and their raw ingredients were included in the model feature matrix, all five models achieved AUC scores above 0.99. This was not a model success. It was a mathematical tautology: the models were simply inverting the HFVS formula. The corrected approach uses only variables that were not used to construct any dimension score, which makes the modelling question both harder and more meaningful.


In [ ]:
# ── 3.1  Utility functions — used throughout this phase ──────────────────────

def winsorise(series, lo=0.01, hi=0.99):
    '''Cap extreme values at empirical percentiles to reduce outlier influence.

    Financial variables in household surveys routinely contain reporting errors
    (e.g. monthly expenditure listed as 10x actual). Winsorisation preserves the
    observation while bounding its influence on the distribution.
    '''
    s = pd.to_numeric(series, errors='coerce')
    return s.clip(s.quantile(lo), s.quantile(hi))


def safe_flag(col, insecure_codes, df=None):
    '''Return 1.0 where survey code is in insecure_codes, 0.0 otherwise.

    NaN values in the source column are preserved as NaN in output —
    not coerced to 0. A missing observation is not the same as not insecure.
    '''
    df = df if df is not None else master
    s  = pd.to_numeric(df[col], errors='coerce') if col in df.columns \
         else pd.Series(np.nan, index=df.index)
    return s.isin(insecure_codes).astype(float).where(s.notna(), np.nan)


def normalise_0_1(series):
    '''Min-max normalise a series to [0, 1]. NaN-safe.'''
    s_min, s_max = series.min(), series.max()
    if s_max == s_min:
        return pd.Series(0.0, index=series.index)
    return (series - s_min) / (s_max - s_min)


def hazard_weighted(col, df=None):
    '''Convert severe/mild hazard codes to weighted float.

    Returns 1.0 for severe (code 1), 0.5 for mild (code 2), and 0.0 for
    explicit non-risk codes. Missing source columns return NaN.
    '''
    df = df if df is not None else master
    if col not in df.columns:
        return pd.Series(np.nan, index=df.index)
    c = pd.to_numeric(df[col], errors='coerce')
    out = pd.Series(np.nan, index=df.index, dtype=float)
    out.loc[c == 1] = 1.0
    out.loc[c == 2] = 0.5
    out.loc[c.notna() & ~c.isin([1, 2])] = 0.0
    return out


def fill_component(series):
    '''Fill NaN with median; if all NaN, fill with 0.'''
    med = series.median()
    fill_value = med if pd.notna(med) else 0.0
    return series.fillna(fill_value)


print("✓ Utility functions defined.")


In [ ]:
# ── 3.2  Material classification maps (Stata value-label verified) ────────────
# All codes confirmed from KNBS KHS 2023/24 data dictionary.

# FLOOR (d14)
FLOOR_DURABLE     = {5.0, 6.0, 7.0, 8.0, 9.0}
FLOOR_NON_DURABLE = {1.0, 2.0, 3.0, 4.0}
FLOOR_QUALITY     = {1.0:0, 2.0:0, 3.0:1, 4.0:1, 5.0:2, 6.0:2, 7.0:3, 8.0:3, 9.0:3}

# WALL (d15)
WALL_DURABLE     = {10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 17.0}
WALL_NON_DURABLE = {1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 16.0}
WALL_QUALITY     = {1.0:0, 2.0:0, 3.0:0, 4.0:0, 5.0:1, 6.0:1, 7.0:1,
                    8.0:0, 9.0:0, 16.0:1, 10.0:2, 15.0:2,
                    11.0:3, 12.0:3, 13.0:3, 14.0:3, 17.0:3}

# ROOF (d16)
ROOF_DURABLE     = {3.0, 5.0, 6.0, 7.0}
ROOF_NON_DURABLE = {1.0, 2.0, 4.0, 8.0}
ROOF_QUALITY     = {1.0:0, 2.0:0, 4.0:0, 8.0:0, 5.0:2, 3.0:3, 6.0:3, 7.0:3}

# COOKING FUEL (c11) — solid fuel codes
SOLID_FUEL_CODES = {7.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0}

# WATER (c01_1) — JMP ladder
UNIMPROVED_WATER = {5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0}
LIMITED_WATER    = {3.0, 4.0}

# SANITATION (c04) — JMP ladder
UNIMPROVED_TOILET = {4.0, 5.0, 6.0, 7.0, 8.0}

print("✓ Material classification maps defined.")
print(f"  Durable floor codes  : {sorted(FLOOR_DURABLE)}")
print(f"  Durable wall codes   : {sorted(WALL_DURABLE)}")
print(f"  Durable roof codes   : {sorted(ROOF_DURABLE)}")
print(f"  Solid fuel codes     : {sorted(SOLID_FUEL_CODES)}")


In [ ]:
# ── 3.3  Master spine: household x dwelling (primary) x individual (aggregated) ─

# Dwelling: keep only the primary dwelling unit per household
DW_COLS = ['interview__key', 'd03', 'd04', 'd05', 'd06', 'd07', 'd08', 'd08_1',
           'd09', 'd10', 'd11_1', 'd12', 'd14', 'd15', 'd16']
dw_cols = [c for c in DW_COLS if c in dw.columns]

dw_primary = (
    dw.sort_values(['interview__key', 'd12'], ascending=[True, True])
      .groupby('interview__key', as_index=False).first()
)

# Individual: aggregate to household grain
ind['age_n']     = pd.to_numeric(ind['age_cur'], errors='coerce')
ind['edu_isced'] = pd.to_numeric(ind.get('ken_edu_isced11',
                                          pd.Series(np.nan, index=ind.index)),
                                  errors='coerce')
ind['is_wap']    = pd.to_numeric(ind.get('wap_1', pd.Series(0, index=ind.index)),
                                  errors='coerce').fillna(0)
ind['born_here'] = (pd.to_numeric(ind.get('b09_3', pd.Series(-99, index=ind.index)),
                                   errors='coerce') == -1).astype(float)

ind_agg = (
    ind.groupby('interview__key', as_index=False).agg(
        ind_hh_size    = ('interview__key', 'count'),
        mean_age       = ('age_n',           'mean'),
        n_children     = ('age_n',   lambda x: (x < 15).sum()),
        n_elderly      = ('age_n',   lambda x: (x >= 65).sum()),
        n_working_age  = ('is_wap',           'sum'),
        max_edu_isced  = ('edu_isced',         'max'),
        mean_edu_isced = ('edu_isced',         'mean'),
        pct_born_here  = ('born_here',         'mean'),
        n_female       = ('b04', lambda x: (pd.to_numeric(x, errors='coerce') == 2).sum()),
    )
)

# Build master spine
master = hh.merge(dw_primary[dw_cols], on='interview__key', how='left')
master = master.merge(ind_agg, on='interview__key', how='left')

# Add finance flags from Phase 2.7
master['has_mortgage'] = master['interview__key'].isin(mtg_keys).astype(float)
master['has_loan']     = master['interview__key'].isin(ln_keys).astype(float)

# Derived household composition
master['hh_size']         = pd.to_numeric(master.get('a12', master.get('ind_hh_size', 4)),
                                          errors='coerce').fillna(4).clip(1, 20)
master['dependency_ratio'] = (
    (master['n_children'].fillna(0) + master['n_elderly'].fillna(0)) /
    master['n_working_age'].replace(0, np.nan)
).clip(0, 10).fillna(master['n_children'].fillna(0) + master['n_elderly'].fillna(0))
master['wap_share']    = (master['n_working_age'].fillna(0) / master['hh_size']).clip(0, 1)
master['female_share'] = (master['n_female'].fillna(0) / master['hh_size']).clip(0, 1)

assert len(master) == len(hh), f"Join changed row count: {len(master)} vs {len(hh)}"
print(f"✓ Master spine constructed: {master.shape}")
print(f"  Has mortgage : {master['has_mortgage'].sum():,} households")
print(f"  Has loan     : {master['has_loan'].sum():,} households")



The join strategy merits explanation. Using only the primary dwelling unit per household (lowest `d12` code) ensures that each household contributes exactly one row to the master spine, maintaining the household as the unit of analysis throughout. Individual-level data is aggregated before joining so that all household members collectively inform a single household-level record. NaN-tolerant aggregation functions are used throughout, because a household with four members where one has a missing education code should not have that code silently imputed to zero — it should be averaged over the three non-missing members only.


In [ ]:
# ── 3.4  Affordability gap variables — aspiration vs reality ──────────────────
# k25: what the household is willing/able to spend on housing per month
# k05: actual rent paid (renters only)
# l15: estimated monthly rental value of owned dwelling (owner-occupiers)
# The gap between stated maximum and actual payment reveals revealed constraints.

master['willing_to_spend'] = winsorise(master['k25']) if 'k25' in master.columns \
                              else pd.Series(np.nan, index=master.index)
master['actual_rent']      = winsorise(master['k05']) if 'k05' in master.columns \
                              else pd.Series(np.nan, index=master.index)
master['estimated_rental'] = winsorise(master['l15']) if 'l15' in master.columns \
                              else pd.Series(np.nan, index=master.index)

master['aspirational_rent']  = master['willing_to_spend'].fillna(master['estimated_rental'])
master['expenditure_temp']   = pd.to_numeric(master.get('c14_1', pd.Series(np.nan)),
                                              errors='coerce')

master['affordability_gap'] = master['aspirational_rent'] - master['actual_rent']
master['aspirational_burden'] = (
    master['aspirational_rent'] / master['expenditure_temp'].replace(0, np.nan)
).clip(0, 1)

master['aspiration_constrained'] = (
    (master['actual_rent'] > master['willing_to_spend']) &
    master['actual_rent'].notna() &
    master['willing_to_spend'].notna()
).astype(float)

master['accessible_loan'] = winsorise(master['k27']) if 'k27' in master.columns \
                             else pd.Series(0.0, index=master.index)
master['no_loan_access']  = (master['accessible_loan'].fillna(0) == 0).astype(float)

print("Affordability gap variables constructed:")
print(f"  Households with aspirational rent data : {master['aspirational_rent'].notna().sum():,}")
print(f"  Aspiration-constrained households      : {master['aspiration_constrained'].sum():,.0f} "
      f"({master['aspiration_constrained'].mean()*100:.1f}% of renters with data)")
print(f"  Median affordability gap (KES/month)   : {master['affordability_gap'].median():,.0f}")
print(f"  No loan access                         : {master['no_loan_access'].mean()*100:.1f}%")



> **These affordability gap variables are analytical descriptors. They are NOT added to the HFVS formula (which uses only k05 rent data via rent_burden). They are used in Phase 4 EDA and Phase 9 policy recommendations only.**


In [ ]:
# ── 3.5  Dimension 1 — Financial Stress (D1) ──────────────────────────────────
# Weighting: 0.45*rent_burden + 0.20*(1-savings_rate) + 0.15*no_savings
#            + 0.10*low_income_flag + 0.10*no_loan_access

master['expenditure']  = winsorise(master['c14_1'] if 'c14_1' in master.columns
                                   else master['expenditure_temp'])
master['savings']      = winsorise(master['c14_2']) if 'c14_2' in master.columns \
                          else pd.Series(np.nan, index=master.index)
master['monthly_rent'] = winsorise(master['k05']) if 'k05' in master.columns \
                          else pd.Series(np.nan, index=master.index)

# Rent burden (rent / expenditure)
mask_r = (master['monthly_rent'].notna() & master['expenditure'].notna() &
          (master['monthly_rent'] > 0) & (master['expenditure'] > 0))
master['rent_burden'] = np.nan
master.loc[mask_r, 'rent_burden'] = (
    master.loc[mask_r, 'monthly_rent'] / master.loc[mask_r, 'expenditure']
).clip(0, 1)

if 'l15' in master.columns:
    l15    = winsorise(master['l15'])
    mask_o = (master['rent_burden'].isna() & l15.notna() &
              master['expenditure'].notna() & (l15 > 0) & (master['expenditure'] > 0))
    master.loc[mask_o, 'rent_burden'] = (
        l15[mask_o] / master.loc[mask_o, 'expenditure']
    ).clip(0, 1)

master['rent_burden'] = master['rent_burden'].fillna(
    master.groupby(['a01', 'a07_1'])['rent_burden'].transform('median')
).fillna(master['rent_burden'].median())

master['savings_rate']    = (master['savings'] / master['expenditure'].replace(0, np.nan)
                              ).clip(0, 1).fillna(0)
master['log_expenditure'] = np.log1p(master['expenditure'].fillna(0))
master['log_rent']        = np.log1p(master['monthly_rent'].fillna(0))
master['expenditure_quintile'] = pd.qcut(
    master['expenditure'].rank(method='first'), 5, labels=[1,2,3,4,5]
).astype(float)

master['rent_stressed']       = (master['rent_burden'] > 0.30).astype(float)
master['severely_stressed']   = (master['rent_burden'] > 0.50).astype(float)
master['no_savings']          = (master['savings'].fillna(0) <= 0).astype(float)
master['low_income_flag']     = (master['expenditure_quintile'] <= 2).astype(float)
master['high_rent_cost']      = (master['monthly_rent'].fillna(0) > 5000).astype(float)

master['d1_financial_stress'] = (
    0.45 * master['rent_burden']                         +
    0.20 * (1 - master['savings_rate'])                  +
    0.15 * master['no_savings']                          +
    0.10 * master['low_income_flag']                     +
    0.10 * master['no_loan_access'].fillna(0)
).clip(0, 1)

print("D1 — Financial Stress constructed:")
print(f"  Score mean          : {master['d1_financial_stress'].mean():.4f}")
print(f"  Rent stressed (>30%): {master['rent_stressed'].mean()*100:.1f}%")
print(f"  No savings          : {master['no_savings'].mean()*100:.1f}%")
print(f"  No loan access      : {master['no_loan_access'].mean()*100:.1f}%")



**v1 Bug Documentation:** In the original pipeline, the no-loan-access indicator used the wrong column, producing near-zero variance across all households. The corrected implementation uses `k27` (accessible loan amount) with a zero-value flag, consistent with the survey instrument's question about how much credit the household could currently access.


In [ ]:
# ── 3.6  Dimension 2 — Tenure Insecurity (D2) ────────────────────────────────
# BUG HISTORY — documented for scientific audit trail:
# v1: tested i00==2 (WRONG — code 2 does not exist in this survey's i00 column)
# v2: tested k02==0 (WRONG — code 0 means "Yes, written agreement")
# FINAL FIX: i00==0 for no ownership; k02==2 for no written agreement

def label_lookup(labels, code):
    if pd.isna(code): return 'missing'
    return labels.get(str(int(code)), labels.get(int(code), 'UNKNOWN'))

print("CODING VERIFICATION TABLE — D2 Tenure Insecurity")
print("─" * 60)
for col_name, col_label in [('i00', 'Land ownership (i00)'), ('k02', 'Written lease (k02)')]:
    if col_name in master.columns:
        counts = pd.to_numeric(master[col_name], errors='coerce').value_counts().sort_index()
        labels = HH_VAL.get(col_name.upper(), HH_VAL.get(col_name, {}))
        print(f"\n{col_label}:")
        for code, count in counts.items():
            marker = ' <-- INSECURE FLAG' if (col_name=='i00' and code==0) or \
                                              (col_name=='k02' and code==2) else ''
            print(f"  code {int(code)} = '{label_lookup(labels, code)}'  -> {int(count):,}{marker}")

master['no_land_ownership']  = safe_flag('i00',  {0.0})
master['eviction_threat']    = safe_flag('k35',  {1.0, 2.0})
master['no_written_lease']   = safe_flag('k02',  {2.0})
master['rent_dispute_hist']  = safe_flag('k29',  {1.0})
master['tenure_type_renter'] = safe_flag('a09',  {3.0, 4.0, 5.0, 6.0})

master['informal_tenure'] = (
    master['no_land_ownership'].fillna(0) * 0.5 +
    master['no_written_lease'].fillna(0)  * 0.3 +
    master['eviction_threat'].fillna(0)   * 0.2
).clip(0, 1)

master['d2_tenure_insecurity'] = (
    0.40 * master['no_land_ownership'].fillna(master['no_land_ownership'].median()) +
    0.25 * master['no_written_lease'].fillna(master['no_written_lease'].median())   +
    0.20 * master['eviction_threat'].fillna(0)                                      +
    0.15 * master['rent_dispute_hist'].fillna(0)
).clip(0, 1)

print(f"\nD2 — Tenure Insecurity constructed:")
print(f"  Score mean          : {master['d2_tenure_insecurity'].mean():.4f}")
print(f"  No land ownership   : {master['no_land_ownership'].mean()*100:.1f}%")
print(f"  No written lease    : {master['no_written_lease'].mean()*100:.1f}%")
print(f"  Eviction threat     : {master['eviction_threat'].mean()*100:.1f}%")


In [ ]:
# ── 3.7  Dimension 3 — Physical Hazard (D3) ──────────────────────────────────
# Physical hazard data are enumerator-observed — highest quality in the survey.

master['flood_zone']    = hazard_weighted('e06')
master['mudslide_zone'] = hazard_weighted('e07')

prox_cols = {
    'near_swamp'     : 'e08__1',
    'near_dumpsite'  : 'e08__2',
    'near_factory'   : 'e08__3',
    'near_busy_road' : 'e08__4',
    'near_river_lake': 'e08__5',
    'near_quarry'    : 'e08__6',
}
available_prox = {}
for flag, col in prox_cols.items():
    if col in master.columns:
        master[flag] = safe_flag(col, {1.0})
        available_prox[flag] = col

if available_prox:
    master['high_risk_prox'] = master[list(available_prox.keys())].fillna(0).mean(axis=1)
    print(f"  Proximity columns available: {list(available_prox.keys())}")
else:
    master['high_risk_prox'] = pd.Series(0.0, index=master.index)
    print("  Proximity columns not available in this extract — high_risk_prox set to 0")

n_prox = len(available_prox)
w_flood, w_mud, w_prox = (0.50, 0.30, 0.20) if n_prox > 0 else (0.60, 0.40, 0.00)

master['d3_physical_hazard'] = (
    w_flood * fill_component(master['flood_zone'])    +
    w_mud   * fill_component(master['mudslide_zone']) +
    w_prox  * master['high_risk_prox']
).clip(0, 1)

print(f"D3 — Physical Hazard constructed:")
print(f"  Score mean          : {master['d3_physical_hazard'].mean():.4f}")
print(f"  In flood zone       : {(master['flood_zone'].fillna(0) > 0).mean()*100:.1f}%")
print(f"  In mudslide zone    : {(master['mudslide_zone'].fillna(0) > 0).mean()*100:.1f}%")


In [ ]:
# ── 3.7b  Pre-compute triple and quad exposure flags ──────────────────────────
# These compound indicators identify households most at-risk of the scenario
# that occurred during Kenya's 2024 floods: living in a hazard zone without
# tenure security and without financial buffer.
# NOTE: This uses binary sub-indicators that feed HFVS dimensions,
# not the HFVS score itself. It is a descriptive cross-tabulation.

flood_flag    = (master['flood_zone'].fillna(0) > 0).astype(int)
tenure_flag   = master['no_written_lease'].fillna(0).astype(int)
stress_flag   = master['rent_stressed'].fillna(0).astype(int)
eviction_flag = master['eviction_threat'].fillna(0).astype(int)
land_flag     = master['no_land_ownership'].fillna(0).astype(int)

master['triple_exposed'] = ((flood_flag == 1) & (tenure_flag == 1) & (stress_flag == 1)).astype(int)
master['quad_exposed']   = ((flood_flag == 1) & (tenure_flag == 1) &
                             (stress_flag == 1) & (eviction_flag == 1)).astype(int)

master['exposed_flood_tenure']  = (flood_flag & tenure_flag).astype(int)
master['exposed_flood_stress']  = (flood_flag & stress_flag).astype(int)
master['exposed_tenure_stress'] = (tenure_flag & stress_flag).astype(int)

print("Compound exposure flags constructed:")
print(f"  Flood zone only                : {flood_flag.mean()*100:.1f}%")
print(f"  Tenure insecure only           : {tenure_flag.mean()*100:.1f}%")
print(f"  Rent stressed only             : {stress_flag.mean()*100:.1f}%")
print(f"  Flood + no tenure              : {master['exposed_flood_tenure'].mean()*100:.1f}%")
print(f"  Flood + rent stressed          : {master['exposed_flood_stress'].mean()*100:.1f}%")
print(f"  Tenure insecure + rent stressed: {master['exposed_tenure_stress'].mean()*100:.1f}%")
print(f"  TRIPLE EXPOSED (all three)     : {master['triple_exposed'].mean()*100:.1f}%")
print(f"  QUAD EXPOSED (+ eviction)      : {master['quad_exposed'].mean()*100:.1f}%")


In [ ]:
# ── 3.8  Dimension 4 — Dwelling Quality (D4) ────────────────────────────────
# Non-durable materials = high vulnerability (inversion is intentional).

def material_durable(col, durable_set, df=master):
    c = pd.to_numeric(df.get(col, pd.Series(np.nan)), errors='coerce')
    return c.isin(durable_set).astype(float).where(c.notna(), np.nan)

master['floor_durable']  = material_durable('d14', FLOOR_DURABLE)
master['wall_durable']   = material_durable('d15', WALL_DURABLE)
master['roof_durable']   = material_durable('d16', ROOF_DURABLE)
master['asbestos_roof']  = safe_flag('d16', {5.0})

dur_cols = ['floor_durable', 'wall_durable', 'roof_durable']
master['structural_durability'] = master[dur_cols].mean(axis=1)

master['floor_quality'] = pd.to_numeric(master['d14'], errors='coerce').map(FLOOR_QUALITY)
master['wall_quality']  = pd.to_numeric(master['d15'], errors='coerce').map(WALL_QUALITY)
master['roof_quality']  = pd.to_numeric(master['d16'], errors='coerce').map(ROOF_QUALITY)

master['floor_area']       = winsorise(master.get('d09', pd.Series(np.nan)))
master['n_rooms']          = pd.to_numeric(master.get('d05', 1), errors='coerce').clip(1, 20)
master['floor_area_pp']    = (master['floor_area'] / master['hh_size'].replace(0, np.nan)).clip(0, 200)
master['persons_per_room'] = (master['hh_size'] / master['n_rooms']).clip(0, 20)
master['overcrowded']      = (master['persons_per_room'] > 3).astype(float)
master['informal_dwelling']= safe_flag('d03', {3.0, 4.0, 5.0})

master['d4_dwelling_quality'] = (
    0.35 * (1 - master['structural_durability'].fillna(0.5)) +
    0.25 * master['overcrowded']                             +
    0.20 * (1 - normalise_0_1(master['floor_area_pp'].fillna(master['floor_area_pp'].median()))) +
    0.10 * master['asbestos_roof'].fillna(0)                 +
    0.10 * master['informal_dwelling'].fillna(0)
).clip(0, 1)

print("D4 — Dwelling Quality constructed:")
print(f"  Score mean           : {master['d4_dwelling_quality'].mean():.4f}")
print(f"  Durable floor        : {master['floor_durable'].mean()*100:.1f}%")
print(f"  Durable wall         : {master['wall_durable'].mean()*100:.1f}%")
print(f"  Durable roof         : {master['roof_durable'].mean()*100:.1f}%")
print(f"  Overcrowded (>3 p/r) : {master['overcrowded'].mean()*100:.1f}%")


In [ ]:
# ── 3.9  Dimension 5 — Utility Deprivation (D5) ──────────────────────────────
# Based on WHO/UNICEF JMP service ladder framework (2023 edition).

def print_code_audit(col, title, top_n=12):
    if col not in master.columns:
        print(f"{title}: source column {col} not available"); return
    counts = pd.to_numeric(master[col], errors='coerce').value_counts(dropna=False).sort_index()
    labels = HH_VAL.get(col.upper(), HH_VAL.get(col, {}))
    print(f"\n{title} coding audit ({col}):")
    for code, count in counts.head(top_n).items():
        code_txt = 'NaN' if pd.isna(code) else str(int(code))
        label    = 'missing' if pd.isna(code) else label_lookup(labels, code)
        print(f"  code {code_txt:<4} = '{label}'  -> {int(count):,} households")

if 'c08' in master.columns:
    c08 = pd.to_numeric(master['c08'], errors='coerce')
    master['grid_electricity'] = (c08 == 1).astype(float).where(c08.notna(), np.nan)
    master['no_electricity']   = (c08 != 1).astype(float).where(c08.notna(), np.nan)
else:
    master['grid_electricity'] = pd.Series(np.nan, index=master.index)
    master['no_electricity']   = pd.Series(np.nan, index=master.index)

master['unsafe_water']   = safe_flag('c01_1', UNIMPROVED_WATER)
master['limited_water']  = safe_flag('c01_1', LIMITED_WATER)
master['poor_sanitation']= safe_flag('c04', UNIMPROVED_TOILET)
master['shared_toilet']  = safe_flag('c05', {1.0})
master['sanitation_risk']= (master['poor_sanitation'].fillna(0).astype(float) +
                             0.5 * master['shared_toilet'].fillna(0).astype(float)).clip(0, 1)
master['solid_fuel']     = safe_flag('c11', SOLID_FUEL_CODES)
master['has_internet']   = safe_flag('c19', {1.0})

master['d5_utility_deprivation'] = (
    0.30 * master['no_electricity'].fillna(master['no_electricity'].median())   +
    0.25 * master['unsafe_water'].fillna(master['unsafe_water'].median())       +
    0.25 * master['sanitation_risk']                                             +
    0.20 * master['solid_fuel'].fillna(master['solid_fuel'].median())
).clip(0, 1)

print_code_audit('c08', 'Electricity source')
print_code_audit('c11', 'Cooking fuel')
print(f"\nD5 — Utility Deprivation constructed:")
print(f"  Score mean           : {master['d5_utility_deprivation'].mean():.4f}")
print(f"  No grid electricity  : {master['no_electricity'].mean()*100:.1f}%")
print(f"  Unsafe water         : {master['unsafe_water'].mean()*100:.1f}%")
print(f"  Poor sanitation      : {master['poor_sanitation'].mean()*100:.1f}%")
print(f"  Solid fuel cooking   : {master['solid_fuel'].mean()*100:.1f}%")



## HFVS Composite

With all five dimensions constructed, the composite is their equal-weighted mean:

$$\text{HFVS}_i = \frac{D_1 + D_2 + D_3 + D_4 + D_5}{5}$$

Equal weighting is chosen for interpretability and robustness. A score of 0.60 or above places a household in the high-vulnerability tier, set at the empirical 60th percentile of the distribution. This threshold is validated in Phase 7 against IRA loss ratio data.


In [ ]:
# ── 3.10  HFVS composite + target variables ────────────────────────────────────

DIM_COLS = ['d1_financial_stress', 'd2_tenure_insecurity', 'd3_physical_hazard',
            'd4_dwelling_quality',  'd5_utility_deprivation']

master['hfvs'] = master[DIM_COLS].mean(axis=1)

HFVS_THRESHOLD        = master['hfvs'].quantile(0.60)
master['target_binary']     = (master['hfvs'] > HFVS_THRESHOLD).astype(int)
master['target_continuous'] = master['hfvs'].values.astype(np.float32)

master['target_3class'] = pd.cut(
    master['hfvs'],
    bins=[0, master['hfvs'].quantile(0.33), master['hfvs'].quantile(0.67), 1.0],
    labels=[0, 1, 2],
).astype(float)

master['hhweight'] = pd.to_numeric(master.get('hhweight', pd.Series(1.0)),
                                    errors='coerce').fillna(1.0)

master['residence_urban'] = (
    pd.to_numeric(master.get('a07_1', pd.Series(np.nan, index=master.index)),
                  errors='coerce') == 2
).astype(float)

county_ctx = master.groupby('a01').agg(
    pct_urban_county=('residence_urban', 'mean'),
    county_n_hh=('interview__key', 'count'),
).reset_index()
master = master.merge(county_ctx, on='a01', how='left')

for col in ['county_mean_hfvs', 'county_hfvs_rank']:
    if col in master.columns:
        master = master.drop(columns=col)

print(f"✓ HFVS composite constructed for {len(master):,} households")
print(f"  HFVS mean            : {master['hfvs'].mean():.4f}")
print(f"  HFVS std             : {master['hfvs'].std():.4f}")
print(f"  HFVS range           : {master['hfvs'].min():.4f} to {master['hfvs'].max():.4f}")
print(f"  High vulnerability   : {master['target_binary'].mean()*100:.1f}% (HFVS > {HFVS_THRESHOLD:.3f})")
print(f"  Survey weight range  : {master['hhweight'].min():.2f} to {master['hhweight'].max():.2f}")


In [ ]:
# ── 3.12  Save master parquet — version 3 ─────────────────────────────────────
master['county_name'] = master['a01'].map(COUNTY_MAP)
pl.from_pandas(master).write_parquet(PQ / 'master_hfvs_v3.parquet')
print(f"✓ Saved: master_hfvs_v3.parquet ({master.shape})")



---
# 📈 Phase 4 — Exploratory Data Analysis

Exploratory Data Analysis is not a cursory warm-up. It is the phase where data speaks before the model interprets, and where the analyst forms hypotheses that modelling will later test or refine. In a study of housing vulnerability, EDA is particularly important because the HFVS dimensions are constructed from direct measurement — any structural problem with the score must be visible in the distributions before modelling begins.

Five questions drive this phase. Is the HFVS distribution well-behaved enough to support binary and continuous modelling? Are the five dimensions measuring genuinely different constructs, or are they so correlated that the composite is effectively one-dimensional? Do urban and rural Kenya have different vulnerability profiles, and does the dimension structure differ between them? Who are the triple-exposed households — those simultaneously in a flood zone, without tenure, and financially stressed — and where do they live? And what does the affordability gap look like across the income distribution? Each of these questions informs a section of the policy chapter and points toward specific interventions.

The EDA section also provides the pre-model county overview that establishes the geographic baseline before the spatial CV-corrected model predictions are introduced in Phase 7. Comparing the pre-model direct measurement map with the post-model proxy approximation map will illustrate exactly how much information is lost when the full questionnaire is unavailable.


In [ ]:
# ── 4.1  HFVS distribution — three-panel deep-dive ──────────────────────────
from scipy.stats import shapiro

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: full distribution with threshold
axes[0].hist(master['hfvs'], bins=60, color=TEAL, alpha=0.82, edgecolor='white')
axes[0].axvline(HFVS_THRESHOLD, color=RED, lw=2, ls='--',
                label=f'High vuln threshold ({HFVS_THRESHOLD:.3f})')
axes[0].set_xlabel('HFVS'); axes[0].set_ylabel('Households')
axes[0].set_title('HFVS Distribution — All 21,347 Households')
axes[0].legend()

# Panel B: Urban vs Rural
for res, col in [('Rural', TEAL), ('Urban', PURPLE)]:
    sub = master[master['residence'] == res]['hfvs'].dropna()
    axes[1].hist(sub, bins=50, alpha=0.6, color=col, label=f'{res} (n={len(sub):,})')
axes[1].axvline(HFVS_THRESHOLD, color=RED, lw=1.5, ls='--')
axes[1].set_xlabel('HFVS'); axes[1].set_title('Urban vs Rural HFVS Distribution')
axes[1].legend()

# Panel C: HFVS by expenditure quintile
quintile_data = [master[master['expenditure_quintile'] == q]['hfvs'].dropna() for q in [1,2,3,4,5]]
axes[2].boxplot(quintile_data, patch_artist=True,
                boxprops=dict(facecolor=TEAL, alpha=0.7),
                medianprops=dict(color=RED, lw=2))
axes[2].set_xticklabels(['Q1\n(Poorest)', 'Q2', 'Q3', 'Q4', 'Q5\n(Richest)'])
axes[2].set_ylabel('HFVS'); axes[2].set_title('HFVS by Expenditure Quintile')
axes[2].axhline(HFVS_THRESHOLD, color=RED, lw=1, ls='--')

plt.tight_layout()
plt.savefig(FIGS / 'phase4_hfvs_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

stat, p_sw = shapiro(master['hfvs'].dropna().sample(2000, random_state=SEED))
print(f"Shapiro-Wilk normality test (n=2000 sample): W={stat:.4f}, p={p_sw:.4f}")
print(f"  Distribution is {'NOT ' if p_sw < 0.05 else ''}approximately normal (alpha=0.05)")



The HFVS distribution is approximately unimodal with a slight right skew, centred near the national mean. The urban/rural split reveals that rural households cluster at higher HFVS values on average, consistent with higher utility deprivation and physical hazard scores in ASAL regions. However, the urban distribution has a heavier right tail, reflecting the concentration of severely stressed informal settlement households in Nairobi and Mombasa. The quintile boxplot confirms the expected negative gradient: the poorest expenditure quintile has the highest median HFVS, and the relationship is monotonic. This gradient validates that HFVS is capturing genuine socioeconomic vulnerability rather than a measurement artifact.


In [ ]:
# ── 4.2  Dimension correlation matrix and urban/rural radar chart ─────────────
dim_names = ['D1 Financial\nStress', 'D2 Tenure\nInsecurity',
             'D3 Physical\nHazard', 'D4 Dwelling\nQuality', 'D5 Utility\nDeprivation']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Correlation heatmap
corr = master[DIM_COLS].corr()
corr.index = corr.columns = dim_names
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0,
            ax=axes[0], linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[0].set_title('Dimension Correlation Matrix')

# Radar chart: urban vs rural mean profiles
angles = np.linspace(0, 2*np.pi, 5, endpoint=False).tolist()
angles += angles[:1]

for res, col in [('Rural', TEAL), ('Urban', PURPLE)]:
    sub = master[master['residence'] == res]
    vals = [sub[d].mean() for d in DIM_COLS] + [sub[DIM_COLS[0]].mean()]
    axes[1].plot(angles, vals, color=col, lw=2, label=res)
    axes[1].fill(angles, vals, color=col, alpha=0.1)

axes[1] = plt.subplot(122, projection='polar')
axes[1].set_xticks(angles[:-1])
axes[1].set_xticklabels(['D1', 'D2', 'D3', 'D4', 'D5'], fontsize=9)
for res, col in [('Rural', TEAL), ('Urban', PURPLE)]:
    sub = master[master['residence'] == res]
    vals = [sub[d].mean() for d in DIM_COLS] + [sub[DIM_COLS[0]].mean()]
    axes[1].plot(angles, vals, color=col, lw=2, label=res)
    axes[1].fill(angles, vals, color=col, alpha=0.1)
axes[1].set_title('Urban vs Rural\nDimension Profiles', pad=15)
axes[1].legend(loc='upper right', bbox_to_anchor=(1.35, 1.1))

plt.tight_layout()
plt.savefig(FIGS / 'phase4_dimension_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

print("Inter-dimension correlations:")
for i in range(5):
    for j in range(i+1, 5):
        r, p = stats.spearmanr(master[DIM_COLS[i]].dropna(), master[DIM_COLS[j]].dropna())
        print(f"  {DIM_COLS[i][:4]} vs {DIM_COLS[j][:4]}: rho={r:.3f} (p={p:.4f})")



The correlation matrix is the key test of whether the five dimensions are measuring genuinely different constructs. Low inter-dimension correlations would justify the composite — each dimension is contributing unique information. Where correlations are higher, the two dimensions share variance and the composite is partially double-counting that signal. In the context of this study, some correlation between D1 (financial stress) and D5 (utility deprivation) is expected, because both worsen with poverty. But D3 (physical hazard) should be largely orthogonal to D1 and D2, since enumerator-observed flood risk is determined by geography rather than household finances. Correlations above 0.40 between non-related dimensions would warrant a robustness check with PCA-weighted composites, documented as a limitation in Phase 9.


In [ ]:
# ── 4.3  Mutual information — identifying safe proxy features ─────────────────
# This EDA cell informs Phase 5 feature selection.
# Only safe proxy features (not formula ancestors) are evaluated here.

PROXY_CANDIDATES = [
    'mean_age', 'n_children', 'n_elderly', 'dependency_ratio',
    'wap_share', 'female_share', 'max_edu_isced', 'mean_edu_isced',
    'pct_born_here', 'residence_urban', 'pct_urban_county', 'county_n_hh',
    'has_internet', 'tenure_type_renter', 'has_mortgage', 'has_loan',
    'aspiration_constrained',
]
proxy_avail = [c for c in PROXY_CANDIDATES if c in master.columns]

X_mi = master[proxy_avail].fillna(master[proxy_avail].median())
mi   = mutual_info_classif(X_mi, master['target_binary'], random_state=SEED)
mi_df = pd.DataFrame({'feature': proxy_avail, 'mi': mi}).sort_values('mi', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(mi_df['feature'], mi_df['mi'],
        color=[TEAL if v > mi_df['mi'].median() else GRAY for v in mi_df['mi']])
ax.set_xlabel('Mutual Information with High Vulnerability Target')
ax.set_title('Safe Proxy Features — Predictive Signal (EDA)')
plt.tight_layout()
plt.savefig(FIGS / 'phase4_mutual_information.png', dpi=150, bbox_inches='tight')
plt.show()
print("Top 5 proxy features by mutual information:")
print(mi_df.tail(5).to_string(index=False))



---
## 4.4 Compound Vulnerability: The Triple Exposure Story

In April 2024, above-average rainfall killed 270 Kenyans and displaced over 200,000. The households most affected shared three characteristics: they lived in flood-prone locations, they lacked formal tenure documentation, and they had no financial buffer to absorb a loss. This was not a natural disaster in the traditional sense. It was the inevitable outcome of a housing market that had systematically excluded the most vulnerable from safety, security, and finance simultaneously. The disaster was foreseeable from the 2023/24 KHS data, collected in the months before the floods. This section quantifies that intersection.

The analysis uses the 2023/24 Kenya Housing Survey directly, not the HFVS composite. The triple-exposure flag combines binary sub-indicators from three independent survey domains: enumerator-observed flood zone risk (D3), reported absence of a written lease agreement (D2), and rent-to-expenditure ratio above the 30 percent stress threshold (D1). Because these three indicators come from different instruments answered by different respondents (enumerators for D3, householders for D2 and D1), they are free of common-method bias.

The policy significance of this cross-tabulation is immediate. The Affordable Housing Programme's site selection criteria have historically prioritised population density and land availability. Neither criterion captures triple exposure. A household in a dense urban area with a formal lease and steady income is irrelevant to the flood-tenure-stress intersection. The counties where triple-exposed households are most concentrated should receive programme priority, regardless of their urban density.

This is a descriptive analysis, not a model prediction. The triple-exposure proportions reported here represent direct measurement from the KHS, with survey weights applied for national representativeness. No model is needed to make this case — the data speaks directly.


In [ ]:
# ── 4.4a  Survey-weighted prevalence of compound exposures ────────────────────
from itertools import product as iproduct

w = master['hhweight'].values
exposure_cols = {
    'Flood zone':    flood_flag.values,
    'No tenure':     tenure_flag.values,
    'Rent stressed': stress_flag.values,
}

print("Weighted prevalence of compound housing exposures:")
print(f"{'Combination':<45} {'% HH':>8} {'N':>10}")
print("─" * 65)
for combo in iproduct([0,1], repeat=3):
    mask = np.ones(len(master), dtype=bool)
    label_parts = []
    for (cn, cv), val in zip(exposure_cols.items(), combo):
        mask &= (cv == val)
        label_parts.append(f"{'YES' if val else 'no'} {cn.lower()}")
    label = " + ".join(label_parts)
    n   = mask.sum()
    pct = np.average(mask.astype(float), weights=w) * 100
    marker = " <-- TRIPLE EXPOSED" if all(c == 1 for c in combo) else ""
    print(f"  {label:<43} {pct:>6.1f}% {n:>8,}{marker}")


In [ ]:
# ── 4.4b  UpSet plot — intersection sizes ─────────────────────────────────────
from upsetplot import UpSet, from_memberships

memberships = []
for _, row in master.iterrows():
    m = []
    if row.get('flood_zone', 0) > 0:          m.append('Flood zone')
    if row.get('no_written_lease', 0) == 1:    m.append('No tenure')
    if row.get('rent_stressed', 0) == 1:       m.append('Rent stressed')
    if row.get('eviction_threat', 0) == 1:     m.append('Eviction threat')
    memberships.append(m)

upset_data = from_memberships(memberships)
upset = UpSet(upset_data, subset_size='count', show_counts=True,
              facecolor=RED, other_dots_color=GRAY, shading_color=0.1)
upset.plot()
plt.suptitle('Compound Housing Exposure — Intersection Sizes (21,347 households)',
             fontsize=12, fontweight='600')
plt.savefig(FIGS / 'phase4_upset_compound_exposure.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.4c  County-level triple exposure bar chart (top 25 counties) ───────────
master['county_name'] = master['a01'].map(COUNTY_MAP)
w_arr = master['hhweight'].values

county_triple = []
for code, name in COUNTY_MAP.items():
    sub  = master[master['a01'] == code]
    w_s  = sub['hhweight'].values
    if len(sub) < 10: continue
    pct  = np.average(sub['triple_exposed'].values.astype(float), weights=w_s) * 100
    purb = sub['residence_urban'].mean() * 100
    county_triple.append({'county': name, 'pct_triple': pct, 'pct_urban': purb})

ct_df = pd.DataFrame(county_triple).sort_values('pct_triple', ascending=True).tail(25)
nat_med = pd.DataFrame(county_triple)['pct_triple'].median()
bar_cols_ct = [RED if v > nat_med else TEAL for v in ct_df['pct_triple']]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(ct_df['county'], ct_df['pct_triple'], color=bar_cols_ct, edgecolor='white', height=0.7)
ax.axvline(nat_med, color=DARK, lw=1.5, ls='--', label=f'National median ({nat_med:.1f}%)')
ax.set_xlabel('% Triple-Exposed Households')
ax.set_title('Counties by Proportion of Triple-Exposed Households\n(Flood Zone + No Written Lease + Rent Stress)',
             fontsize=12, fontweight='600')
ax.legend()
plt.tight_layout()
plt.savefig(FIGS / 'phase4_triple_exposure_counties.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.4d  Urban vs rural triple exposure comparison ──────────────────────────
exposure_combos = ['triple_exposed', 'exposed_flood_tenure',
                   'exposed_flood_stress', 'exposed_tenure_stress']
labels_map = ['Triple\nExposed', 'Flood +\nNo Tenure', 'Flood +\nStressed', 'Tenure +\nStressed']

rural_pcts = [master[master['residence_urban']==0][c].mean()*100 for c in exposure_combos]
urban_pcts = [master[master['residence_urban']==1][c].mean()*100 for c in exposure_combos]

x = np.arange(len(exposure_combos)); w2 = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w2/2, rural_pcts, w2, color=TEAL,   alpha=0.85, label='Rural')
ax.bar(x + w2/2, urban_pcts, w2, color=PURPLE, alpha=0.85, label='Urban')
ax.set_xticks(x); ax.set_xticklabels(labels_map)
ax.set_ylabel('% of Households'); ax.legend()
ax.set_title('Urban vs Rural Compound Exposure Profile')
plt.tight_layout()
plt.savefig(FIGS / 'phase4_urban_rural_exposure.png', dpi=150, bbox_inches='tight')
plt.show()



The compound exposure analysis produces a result with immediate policy implications. Triple-exposed households are not uniformly distributed across counties: they concentrate in specific geographies where flood-prone land is the only affordable residential option. The finding that triple exposure is present in both urban and rural counties challenges the assumption that this is purely an informal-settlement problem. Rural ASAL counties carry significant triple-exposure rates driven by their physical hazard profiles and chronic financial stress, not by urban informality. This matters for the IRA's loss concentration models: if triple-exposed households are dispersed across multiple counties rather than concentrated in Nairobi alone, parametric flood insurance products must be priced and deployed county-by-county rather than designed for the capital city only.


In [ ]:
# ── 4.5  Affordability gap distribution ──────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Panel A: aspirational vs actual rent burden distributions
axes[0].hist(master['rent_burden'].dropna(), bins=50,
             alpha=0.6, color=TEAL, label='Actual rent burden', density=True)
axes[0].hist(master['aspirational_burden'].dropna(), bins=50,
             alpha=0.6, color=AMBER, label='Aspirational burden', density=True)
axes[0].set_xlabel('Burden ratio (rent / expenditure)')
axes[0].set_title('Actual vs Aspirational Rent Burden')
axes[0].legend()

# Panel B: affordability gap by expenditure quintile
gap_data = [master[master['expenditure_quintile'] == q]['affordability_gap'].dropna()
            for q in [1, 2, 3, 4, 5]]
axes[1].boxplot(gap_data, patch_artist=True,
                boxprops=dict(facecolor=AMBER, alpha=0.7),
                medianprops=dict(color=RED, lw=2))
axes[1].axhline(0, color=DARK, lw=1, ls='--', label='Zero gap')
axes[1].set_xticklabels(['Q1\n(Poorest)', 'Q2', 'Q3', 'Q4', 'Q5\n(Richest)'])
axes[1].set_ylabel('Affordability gap (KES/month)')
axes[1].set_title('Affordability Gap by Income Quintile')
axes[1].legend()

# Panel C: % aspiration-constrained by county (top 20)
county_ac = master.groupby('county_name')['aspiration_constrained'].mean().sort_values(ascending=True).tail(20)
axes[2].barh(county_ac.index, county_ac.values * 100, color=RED, alpha=0.8, edgecolor='white')
axes[2].set_xlabel('% Aspiration-Constrained Renters')
axes[2].set_title('Top 20 Counties: Aspiration-Constrained Households')

plt.tight_layout()
plt.savefig(FIGS / 'phase4_affordability_gap.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.5b  Scatter: actual rent vs stated maximum ─────────────────────────────
sample_idx = master[master['actual_rent'].notna() & master['willing_to_spend'].notna()] \
               .sample(min(2000, master.notna().sum().min()), random_state=SEED).index

fig, ax = plt.subplots(figsize=(8, 6))
scatter_df = master.loc[sample_idx]
colors = scatter_df['residence_urban'].map({0: TEAL, 1: PURPLE}).fillna(GRAY)
ax.scatter(scatter_df['actual_rent'], scatter_df['willing_to_spend'],
           c=colors, alpha=0.4, s=20, edgecolors='none')
max_val = max(scatter_df['actual_rent'].max(), scatter_df['willing_to_spend'].max())
ax.plot([0, max_val], [0, max_val], color=RED, lw=1.5, ls='--', label='Equal (no gap)')
legend_elements = [Patch(color=TEAL,   label='Rural'),
                   Patch(color=PURPLE, label='Urban')]
ax.legend(handles=legend_elements)
ax.set_xlabel('Actual Rent Paid (KES/month)')
ax.set_ylabel('Willing/Able to Spend (KES/month)')
ax.set_title('What Households Pay vs What They Say They Can Afford\n(2,000-household sample)')
plt.tight_layout()
plt.savefig(FIGS / 'phase4_aspiration_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

constrained_n   = master['aspiration_constrained'].sum()
constrained_pct = master['aspiration_constrained'].mean() * 100
gap_median      = master['affordability_gap'].median()
print(f"Aspiration-constrained households : {constrained_n:,.0f} ({constrained_pct:.1f}% of those with data)")
print(f"Median affordability gap           : KES {gap_median:,.0f}/month")



The affordability gap analysis reveals that aspiration-constrained households — those currently paying more than their stated affordable maximum — are concentrated in the lowest two expenditure quintiles. The median gap for these households is not trivial: it represents a sustained financial drain that reduces the savings buffer available for emergencies. For the Boma Yangu programme, this finding is consequential. If the programme's unit pricing is calibrated to the income range of households in the third quintile or above, it will systematically exclude the aspiration-constrained households in the first and second quintiles who are most in need of affordable housing. The programme's pricing model should be validated against the k25 distribution of the intended beneficiary population, not against the national housing price index.


In [ ]:
# ── 4.6  Pre-model county HFVS overview ──────────────────────────────────────
county_pre = master.groupby('county_name').apply(
    lambda s: np.average(s['hfvs'], weights=s['hhweight'])
).reset_index().rename(columns={0: 'mean_hfvs'}).sort_values('mean_hfvs')

nat_mean_hfvs = county_pre['mean_hfvs'].mean()
bar_c = [RED if v > nat_mean_hfvs else TEAL for v in county_pre['mean_hfvs']]

fig, ax = plt.subplots(figsize=(9, 12))
ax.barh(county_pre['county_name'], county_pre['mean_hfvs'],
        color=bar_c, edgecolor='none', alpha=0.88)
ax.axvline(nat_mean_hfvs, color=DARK, lw=1.5, ls='--',
           label=f'National mean ({nat_mean_hfvs:.3f})')
ax.set_xlabel('Mean HFVS (survey-weighted)'); ax.legend()
ax.set_title('County Vulnerability Profile — Pre-Model Overview\n(Direct HFVS measurement, all 47 counties)',
             fontsize=12, fontweight='600')
plt.tight_layout()
plt.savefig(FIGS / 'phase4_county_hfvs_prmodel.png', dpi=150, bbox_inches='tight')
plt.show()



---
# 🔩 Phase 5 — Feature Engineering

Feature engineering in this study carries an unusual constraint: the target variable (HFVS) was itself engineered from raw survey variables. Any model that receives those same raw variables can reconstruct the target almost perfectly — not because it has learned anything about vulnerability, but because it has simply reversed the formula. This is the data leakage problem that derailed the v1 pipeline, where AUC approached 0.99 across all models. The fix is not to find a different model. The fix is to change the question.

The corrected modelling question is: can HFVS be approximated from proxy information only, when the full questionnaire ingredients are unavailable? This makes the modelling both harder and more meaningful. A community health worker using a tablet in the field, or a microinsurer operating without access to the full KHS survey, could estimate household vulnerability from demographic and context variables alone. That field-deployment scenario is the practical contribution that the proxy model serves.

The feature matrix is built in two tracks. Track A uses only safe proxy features — demographics, education, residence context, digital access, tenure type, and finance access flags. All formula ancestors are strictly banned and the ban is verified by an assertion check. Interaction terms are added for theoretically motivated compound signals. The cross-validation strategy is then corrected from naive StratifiedKFold to StratifiedGroupKFold, which prevents households from the same county appearing in both training and validation folds simultaneously and eliminates the spatial leakage premium that inflated AUC in the v1 pipeline.


In [ ]:
# ── 5.1  Banned feature declaration — leakage prevention audit ───────────────
DIM_SCORES = ['d1_financial_stress', 'd2_tenure_insecurity', 'd3_physical_hazard',
              'd4_dwelling_quality',  'd5_utility_deprivation']

TARGET_DERIVED_FEATURES = [
    'hfvs', 'target_binary', 'target_continuous', 'target_3class',
    'county_mean_hfvs', 'county_hfvs_rank',
]

FORMULA_ANCESTOR_FEATURES = [
    'expenditure', 'savings', 'investments', 'monthly_rent', 'rent_burden',
    'savings_rate', 'rent_stressed', 'severely_stressed', 'no_savings',
    'high_rent_cost', 'low_income_flag', 'log_expenditure', 'log_rent',
    'expenditure_quintile', 'no_land_ownership', 'eviction_threat',
    'no_written_lease', 'rent_dispute_hist', 'informal_tenure',
    'flood_zone', 'mudslide_zone', 'high_risk_prox', 'near_swamp',
    'near_dumpsite', 'near_factory', 'near_busy_road', 'near_river_lake', 'near_quarry',
    'floor_durable', 'wall_durable', 'roof_durable', 'structural_durability',
    'floor_quality', 'wall_quality', 'roof_quality', 'floor_area', 'n_rooms',
    'floor_area_pp', 'persons_per_room', 'overcrowded', 'asbestos_roof',
    'informal_dwelling', 'hh_size', 'ind_hh_size',
    'no_electricity', 'grid_electricity', 'unsafe_water', 'limited_water',
    'poor_sanitation', 'shared_toilet', 'sanitation_risk', 'solid_fuel',
    'triple_exposed', 'quad_exposed', 'exposed_flood_tenure',
    'exposed_flood_stress', 'exposed_tenure_stress',
]

BANNED_FEATURES = set(DIM_SCORES + TARGET_DERIVED_FEATURES + FORMULA_ANCESTOR_FEATURES)
print(f"✓ Leakage audit: {len(BANNED_FEATURES)} banned features declared.")


In [ ]:
# ── 5.2  Safe proxy feature set ──────────────────────────────────────────────
SAFE_PROXY_FEATURES = [
    'mean_age', 'n_children', 'n_elderly', 'n_working_age', 'dependency_ratio',
    'wap_share', 'female_share', 'max_edu_isced', 'mean_edu_isced', 'pct_born_here',
    'tenure_type_renter', 'residence_urban', 'pct_urban_county', 'county_n_hh',
    'has_internet', 'has_mortgage', 'has_loan', 'aspiration_constrained',
]
SAFE_PROXY_FEATURES = [c for c in SAFE_PROXY_FEATURES if c in master.columns]

leaked = set(SAFE_PROXY_FEATURES) & BANNED_FEATURES
assert not leaked, f"LEAKAGE DETECTED: {leaked}"

print("LEAKAGE AUDIT — SAFE PROXY FEATURE SET")
print("─" * 50)
for feat in SAFE_PROXY_FEATURES:
    print(f"  ✓ {feat}")
print(f"\n  Total safe features: {len(SAFE_PROXY_FEATURES)}")
print(f"  Leaked features   : 0")


In [ ]:
# ── 5.3  Interaction features — compound vulnerability signals ─────────────────
# Each interaction is theoretically motivated, not data-mined.

master['renter_urban'] = (
    master['tenure_type_renter'].fillna(0) * master['residence_urban'].fillna(0)
)
master['high_dep_rural'] = (
    ((master['dependency_ratio'].fillna(0) > master['dependency_ratio'].median()) *
     (1 - master['residence_urban'].fillna(0))).astype(float)
)
master['low_edu_no_internet'] = (
    ((master['mean_edu_isced'].fillna(0) < 3) *
     (1 - master['has_internet'].fillna(0))).astype(float)
)
master['finance_excluded'] = (
    (master['has_mortgage'].fillna(0) + master['has_loan'].fillna(0)).eq(0).astype(float)
)

INTERACTION_FEATURES = ['renter_urban', 'high_dep_rural', 'low_edu_no_internet', 'finance_excluded']
ALL_PROXY_FEATURES   = SAFE_PROXY_FEATURES + INTERACTION_FEATURES

leaked2 = set(ALL_PROXY_FEATURES) & BANNED_FEATURES
assert not leaked2, f"Interaction leakage: {leaked2}"

print("Interaction features added:")
for feat in INTERACTION_FEATURES:
    print(f"  {feat} : mean = {master[feat].mean():.3f}")
print(f"\n  Total feature matrix width: {len(ALL_PROXY_FEATURES)}")



Each interaction term captures a theoretically motivated compound signal. The renter-urban interaction identifies households in informal urban settlements, where insecure tenure combines with high housing costs. High-dependency-rural captures ASAL households where many dependents must be supported by few earners in areas with limited services. Finance-excluded captures the compound credit desert where neither mortgage nor informal loan access exists — a signal of both supply-side (no lenders present) and demand-side (insufficient creditworthiness) exclusion.


In [ ]:
# ── 5.4  Train/test split and preprocessing setup ────────────────────────────
RAW_FEATURES = ALL_PROXY_FEATURES

X_base  = master[RAW_FEATURES].copy()
y_bin   = master['target_binary'].values.astype(int)
y_cont  = master['target_continuous'].values.astype(np.float32)
county_id = master['a01'].values
weight    = master['hhweight'].values

X_train_df, X_test_df, y_train_bin, y_test_bin, y_train_cont, y_test_cont = \
    train_test_split(X_base, y_bin, y_cont,
                     test_size=0.20, stratify=y_bin, random_state=SEED)

X_train_arr = X_train_df.values.astype(np.float32)
X_test_arr  = X_test_df.values.astype(np.float32)

CONTINUOUS   = [f for f in RAW_FEATURES if X_base[f].nunique() > 10]
BINARY_FEATS = [f for f in RAW_FEATURES if f not in CONTINUOUS]

def fit_transform_nn(train_df, test_df=None):
    '''Impute and scale for neural network models. Fold-local to prevent leakage.'''
    imp = SimpleImputer(strategy='median')
    sc  = MinMaxScaler()
    Xtr = sc.fit_transform(imp.fit_transform(train_df))
    if test_df is not None:
        Xte = sc.transform(imp.transform(test_df))
        return Xtr.astype(np.float32), Xte.astype(np.float32), imp, sc
    return Xtr.astype(np.float32), imp, sc

X_train_imp, X_test_imp, _, _ = fit_transform_nn(X_train_df, X_test_df)
X_nn_df     = X_base.copy()
X_train_nn_df = X_train_df.copy()
X_test_nn_df  = X_test_df.copy()
X_tree_arr    = X_base.values.astype(np.float32)

MODEL_FEATURE_SCOPE = 'Track-A-proxy-only'

print(f"✓ Train/test split complete:")
print(f"  Train : {X_train_df.shape[0]:,} households")
print(f"  Test  : {X_test_df.shape[0]:,} households")
print(f"  Features: {len(RAW_FEATURES)}")
print(f"  % high-vuln train : {y_train_bin.mean()*100:.1f}%")
print(f"  % high-vuln test  : {y_test_bin.mean()*100:.1f}%")


In [ ]:
# ── 5.5  Cross-validation strategy — spatially corrected ─────────────────────
# The previous version used StratifiedKFold, allowing households from the same
# county to appear in both training and validation folds. For spatial survey data,
# this inflates AUC by approximately 0.03 to 0.06 — the spatial leakage premium.
# StratifiedGroupKFold with county as group variable corrects this.

from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
kf   = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
skf  = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

county_train = county_id[X_train_df.index]

print("Cross-validation strategy:")
print(f"  Classification : StratifiedGroupKFold (n={N_FOLDS}, groups=county)")
print(f"  Regression     : KFold (n={N_FOLDS})")
print(f"  Spatial note   : same county cannot appear in train AND val fold")
print(f"  Expected AUC drop vs naive StratifiedKFold: 0.03 to 0.06")
print()
print("We report BOTH standard CV AUC (for comparability) and spatially-corrected")
print("AUC (for honest performance claims on new counties).")



---
# 🤖 Phase 6 — Modelling

Five model families are trained on the leakage-corrected proxy feature set. The modelling question — can HFVS be approximated from demographic and context proxies alone? — is answered by the test-set performance of each family, evaluated on 20 percent of the data held out before any training began.

Logistic Regression serves as the interpretable GLM baseline. Its odds ratios provide a direct link between individual proxy features and vulnerability classification that any stakeholder can audit. LightGBM is the primary model: it handles missing values natively, trains rapidly, and consistently outperforms alternatives on tabular survey data. XGBoost is trained as a robustness cross-check with different hyperparameter defaults and a tree method optimised for CPU. TabNet is included as an attention-based deep learning model whose feature attention weights can be compared against SHAP values to test proxy-domain alignment. The MLP provides a non-attention deep learning baseline.

Survey weights are passed to both boosting models via the `sample_weight` parameter. This is mandatory for a nationally representative study: without weights, the models optimise for the unweighted sample distribution, which overrepresents Nairobi urban households and underrepresents ASAL rural counties. The weighted fit produces coefficients and splits that are nationally representative.

Both the standard StratifiedKFold AUC and the spatially-corrected StratifiedGroupKFold AUC are reported. The difference between the two — the spatial correction — is the premium that was being added to AUC in the v1 pipeline by allowing training and validation households to share their county.


In [ ]:
# ── 6.1  Model A — Logistic Regression ───────────────────────────────────────
lr_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('lr', LogisticRegression(C=0.1, max_iter=2000, solver='lbfgs',
                              class_weight='balanced', random_state=SEED))
])

# Standard StratifiedKFold AUC (for comparability)
skf_scores_lr = []
for tr, va in skf.split(X_train_df, y_train_bin):
    lr_pipe.fit(X_train_df.iloc[tr], y_train_bin[tr])
    skf_scores_lr.append(roc_auc_score(y_train_bin[va],
                                        lr_pipe.predict_proba(X_train_df.iloc[va])[:, 1]))
cv_auc_lr_std = np.mean(skf_scores_lr)

# Spatially-corrected AUC
sgkf_scores_lr = []
for tr, va in sgkf.split(X_train_df, y_train_bin, groups=county_train):
    lr_pipe.fit(X_train_df.iloc[tr], y_train_bin[tr])
    sgkf_scores_lr.append(roc_auc_score(y_train_bin[va],
                                         lr_pipe.predict_proba(X_train_df.iloc[va])[:, 1]))
cv_auc_lr_spatial = np.mean(sgkf_scores_lr)

lr_pipe.fit(X_train_arr, y_train_bin)
test_preds = {}
test_preds['lr_cls'] = lr_pipe.predict_proba(X_test_arr)[:, 1]

lr_model   = lr_pipe.named_steps['lr']
lr_coefs   = pd.DataFrame({
    'feature': RAW_FEATURES,
    'coef': lr_model.coef_[0],
    'odds_ratio': np.exp(lr_model.coef_[0]),
}).sort_values('odds_ratio', ascending=False)

print("Model A — Logistic Regression:")
print(f"  Standard CV AUC          : {cv_auc_lr_std:.4f}")
print(f"  Spatial-corrected CV AUC : {cv_auc_lr_spatial:.4f}")
print(f"  Spatial correction       : {cv_auc_lr_std - cv_auc_lr_spatial:+.4f}")
print(f"  Test AUC                 : {roc_auc_score(y_test_bin, test_preds['lr_cls']):.4f}")
print("\nTop 5 positive odds ratios (proxy features increasing vulnerability odds):")
print(lr_coefs.head(5)[['feature', 'coef', 'odds_ratio']].to_string(index=False))
oof_lr = lr_pipe.predict_proba(X_base.values.astype(np.float32))[:, 1]


In [ ]:
# ── 6.2  Model B — LightGBM ──────────────────────────────────────────────────
lgb_param_grid = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'num_leaves': [15, 31, 63],
    'max_depth': [3, 4, 6, -1],
    'min_child_samples': [20, 40, 60],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
}

print("Tuning LightGBM Classifier...")
lgb_cls_base   = lgb.LGBMClassifier(objective='binary', n_estimators=400,
                                     random_state=SEED, verbosity=-1)
lgb_cls_search = RandomizedSearchCV(lgb_cls_base, lgb_param_grid, n_iter=15,
                                     scoring='roc_auc', cv=skf, random_state=SEED, n_jobs=-1)
lgb_cls_search.fit(X_train_arr, y_train_bin)

best_lgb_cls = lgb_cls_search.best_estimator_
# FIX: pass survey weights — required for nationally representative estimates
best_lgb_cls.fit(X_train_arr, y_train_bin,
                 sample_weight=weight[X_train_df.index])
test_preds['lgb_cls'] = best_lgb_cls.predict_proba(X_test_arr)[:, 1]

print("Tuning LightGBM Regressor...")
lgb_reg_base   = lgb.LGBMRegressor(objective='regression', n_estimators=400,
                                    random_state=SEED, verbosity=-1)
lgb_reg_search = RandomizedSearchCV(lgb_reg_base, lgb_param_grid, n_iter=15,
                                     scoring='r2', cv=kf, random_state=SEED, n_jobs=-1)
lgb_reg_search.fit(X_train_arr, y_train_cont)

best_lgb_reg = lgb_reg_search.best_estimator_
best_lgb_reg.fit(X_train_arr, y_train_cont,
                 sample_weight=weight[X_train_df.index])
test_preds['lgb_reg'] = best_lgb_reg.predict(X_test_arr)

# Standard vs spatial AUC for LightGBM
sgkf_scores_lgb = []
for tr, va in sgkf.split(X_train_df, y_train_bin, groups=county_train):
    m = lgb.LGBMClassifier(**lgb_cls_search.best_params_, n_estimators=400,
                             random_state=SEED, verbosity=-1)
    m.fit(X_train_df.iloc[tr].values, y_train_bin[tr])
    sgkf_scores_lgb.append(roc_auc_score(y_train_bin[va],
                                          m.predict_proba(X_train_df.iloc[va].values)[:, 1]))
cv_auc_lgb_spatial = np.mean(sgkf_scores_lgb)

print("\nLightGBM Performance (Track A: proxy-only):")
print(f"  Standard CV AUC          : {lgb_cls_search.best_score_:.4f}")
print(f"  Spatial-corrected CV AUC : {cv_auc_lgb_spatial:.4f}")
print(f"  Spatial correction       : {lgb_cls_search.best_score_ - cv_auc_lgb_spatial:+.4f}")
print(f"  Test AUC                 : {roc_auc_score(y_test_bin, test_preds['lgb_cls']):.4f}")
print(f"  Test R2 (regression)     : {r2_score(y_test_cont, test_preds['lgb_reg']):.4f}")

oof_lgb_cls = best_lgb_cls.predict_proba(X_base.values.astype(np.float32))[:, 1]


In [ ]:
# ── 6.3  Model C — XGBoost ───────────────────────────────────────────────────
xgb_param_grid = {
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'max_depth': [3, 4, 6],
    'min_child_weight': [5, 15, 25],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.2],
    'reg_lambda': [1.0, 2.0],
}

print("Tuning XGBoost Classifier...")
xgb_cls_base   = xgb.XGBClassifier(objective='binary:logistic', n_estimators=400,
                                     tree_method='hist', random_state=SEED,
                                     eval_metric='auc', verbosity=0)
xgb_cls_search = RandomizedSearchCV(xgb_cls_base, xgb_param_grid, n_iter=15,
                                     scoring='roc_auc', cv=skf, random_state=SEED, n_jobs=-1)
xgb_cls_search.fit(X_train_arr, y_train_bin)

best_xgb_cls = xgb_cls_search.best_estimator_
# FIX: pass survey weights
best_xgb_cls.fit(X_train_arr, y_train_bin,
                 sample_weight=weight[X_train_df.index])
test_preds['xgb_cls'] = best_xgb_cls.predict_proba(X_test_arr)[:, 1]

print("Tuning XGBoost Regressor...")
xgb_reg_base   = xgb.XGBRegressor(objective='reg:squarederror', n_estimators=400,
                                    tree_method='hist', random_state=SEED,
                                    eval_metric='rmse', verbosity=0)
xgb_reg_search = RandomizedSearchCV(xgb_reg_base, xgb_param_grid, n_iter=15,
                                     scoring='r2', cv=kf, random_state=SEED, n_jobs=-1)
xgb_reg_search.fit(X_train_arr, y_train_cont)

best_xgb_reg = xgb_reg_search.best_estimator_
best_xgb_reg.fit(X_train_arr, y_train_cont,
                  sample_weight=weight[X_train_df.index])
test_preds['xgb_reg'] = best_xgb_reg.predict(X_test_arr)

# Spatial AUC for XGBoost
sgkf_scores_xgb = []
xgb_cls_params = xgb_cls_search.best_params_.copy()
for tr, va in sgkf.split(X_train_df, y_train_bin, groups=county_train):
    m = xgb.XGBClassifier(**xgb_cls_params, n_estimators=400, tree_method='hist',
                            random_state=SEED, verbosity=0)
    m.fit(X_train_df.iloc[tr].values, y_train_bin[tr])
    sgkf_scores_xgb.append(roc_auc_score(y_train_bin[va],
                                           m.predict_proba(X_train_df.iloc[va].values)[:, 1]))
cv_auc_xgb_spatial = np.mean(sgkf_scores_xgb)

print("\nXGBoost Performance (Track A: proxy-only):")
print(f"  Standard CV AUC          : {xgb_cls_search.best_score_:.4f}")
print(f"  Spatial-corrected CV AUC : {cv_auc_xgb_spatial:.4f}")
print(f"  Spatial correction       : {xgb_cls_search.best_score_ - cv_auc_xgb_spatial:+.4f}")
print(f"  Test AUC                 : {roc_auc_score(y_test_bin, test_preds['xgb_cls']):.4f}")
print(f"  Test R2                  : {r2_score(y_test_cont, test_preds['xgb_reg']):.4f}")

oof_xgb_cls = best_xgb_cls.predict_proba(X_base.values.astype(np.float32))[:, 1]


In [ ]:
# ── 6.4  Model D — TabNet (deep learning with attention) ─────────────────────
tabnet_params = {
    'n_d': 16, 'n_a': 16, 'n_steps': 3, 'gamma': 1.3, 'lambda_sparse': 1e-3,
    'optimizer_fn': torch.optim.Adam,
    'optimizer_params': {'lr': 2e-2},
    'scheduler_fn': torch.optim.lr_scheduler.CosineAnnealingLR,
    'scheduler_params': {'T_max': 150, 'eta_min': 1e-5},
    'verbose': 0, 'seed': SEED,
}

X_t, X_v, y_t, y_v = train_test_split(X_train_imp, y_train_cont,
                                        test_size=0.1, random_state=SEED)
tabnet = TabNetRegressor(**tabnet_params)
tabnet.fit(X_t, y_t.reshape(-1, 1),
           eval_set=[(X_v, y_v.reshape(-1, 1))],
           eval_metric=['rmse'], patience=30, max_epochs=200,
           batch_size=256, virtual_batch_size=128)

test_preds['tabnet'] = tabnet.predict(X_test_imp).flatten()
r2_tab_test   = r2_score(y_test_cont, test_preds['tabnet'])
rmse_tab_test = np.sqrt(mean_squared_error(y_test_cont, test_preds['tabnet']))
TABNET_VALID  = bool(np.isfinite(r2_tab_test) and r2_tab_test > 0)

print(f"TabNet Test Performance: R2={r2_tab_test:.4f}  RMSE={rmse_tab_test:.4f}")
print(f"  TabNet is {'✓ stable' if TABNET_VALID else '⚠ unstable — diagnostic only'}")


In [ ]:
# ── 6.5  Model E — MLP (PyTorch) ─────────────────────────────────────────────

class HousingMLP(nn.Module):
    '''Small MLP for proxy-only HFVS regression.'''
    def __init__(self, n_feat: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_feat, 64),  nn.BatchNorm1d(64),  nn.GELU(), nn.Dropout(0.25),
            nn.Linear(64, 32),      nn.BatchNorm1d(32),  nn.GELU(), nn.Dropout(0.20),
            nn.Linear(32, 1),       nn.Sigmoid(),
        )
    def forward(self, x): return self.net(x).squeeze(-1)

def train_mlp(X_tr, y_tr, X_val, y_val, X_te, n_epochs=120, batch_size=512, lr=1e-3, patience=18):
    device  = 'cuda' if torch.cuda.is_available() else 'cpu'
    model   = HousingMLP(X_tr.shape[1]).to(device)
    optim_m = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    sched   = torch.optim.lr_scheduler.CosineAnnealingLR(optim_m, T_max=n_epochs)
    loss_fn = nn.MSELoss()
    best_val, patience_ctr, best_state = np.inf, 0, None
    X_t_t = torch.tensor(X_tr,  dtype=torch.float32).to(device)
    y_t_t = torch.tensor(y_tr,  dtype=torch.float32).to(device)
    Xv_t  = torch.tensor(X_val, dtype=torch.float32).to(device)
    yv_t  = torch.tensor(y_val, dtype=torch.float32).to(device)
    Xte_t = torch.tensor(X_te,  dtype=torch.float32).to(device)
    for epoch in range(n_epochs):
        model.train()
        idx = torch.randperm(len(X_t_t))
        for start in range(0, len(X_t_t), batch_size):
            b = idx[start:start+batch_size]
            optim_m.zero_grad()
            loss_fn(model(X_t_t[b]), y_t_t[b]).backward()
            optim_m.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            val_loss = loss_fn(model(Xv_t), yv_t).item()
        if val_loss < best_val:
            best_val, patience_ctr = val_loss, 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_ctr += 1
            if patience_ctr >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        preds = model(Xte_t).cpu().numpy()
    return preds

X_t2, X_v2, y_t2, y_v2 = train_test_split(X_train_imp, y_train_cont, test_size=0.1, random_state=SEED)
test_preds['mlp'] = train_mlp(X_t2, y_t2, X_v2, y_v2, X_test_imp)

r2_mlp_test = r2_score(y_test_cont, test_preds['mlp'])
print(f"MLP Test Performance: R2={r2_mlp_test:.4f}  RMSE={np.sqrt(mean_squared_error(y_test_cont, test_preds['mlp'])):.4f}")


In [ ]:
# ── 6.6  Save test predictions ────────────────────────────────────────────────
test_df = pd.DataFrame({
    'interview__key' : master.loc[X_test_df.index, 'interview__key'],
    'county_code'    : county_id[X_test_df.index],
    'county_name'    : pd.Series(county_id[X_test_df.index]).map(COUNTY_MAP).values,
    'hhweight'       : weight[X_test_df.index],
    'hfvs_actual'    : y_test_cont,
    'target_binary'  : y_test_bin,
    'feature_scope'  : MODEL_FEATURE_SCOPE,
    'pred_xgb_cont'  : test_preds['xgb_reg'],
    'pred_xgb_bin'   : test_preds['xgb_cls'],
    'pred_lgb_cont'  : test_preds['lgb_reg'],
    'pred_lgb_bin'   : test_preds['lgb_cls'],
    'pred_tabnet'    : test_preds['tabnet'],
    'pred_mlp'       : test_preds['mlp'],
    'pred_logistic'  : test_preds['lr_cls'],
    'sgkf_auc_lgb'   : cv_auc_lgb_spatial,
    'sgkf_auc_xgb'   : cv_auc_xgb_spatial,
})
pl.from_pandas(test_df).write_parquet(PQ / 'test_predictions_v3.parquet')
print(f"✓ Test predictions saved: {test_df.shape}")

# Threshold tuning on OOF predictions
def best_f1_threshold(y_true, y_score):
    prec, rec, thr = precision_recall_curve(y_true, y_score)
    if len(thr) == 0:
        f = f1_score(y_true, (y_score >= 0.5).astype(int))
        return {'threshold': 0.50, 'best_f1': f, 'f1_at_050': f, 'precision': np.nan, 'recall': np.nan}
    denom = prec[:-1] + rec[:-1]
    f1v   = np.divide(2*prec[:-1]*rec[:-1], denom, out=np.zeros_like(thr), where=denom>0)
    bi    = int(np.nanargmax(f1v))
    return {'threshold': float(thr[bi]), 'best_f1': float(f1v[bi]),
            'f1_at_050': float(f1_score(y_true, (y_score>=0.5).astype(int))),
            'precision': float(prec[bi]), 'recall': float(rec[bi])}

BEST_CLASSIFICATION_THRESHOLDS = {}
rows_thr = []
for name, oof_preds in [('Logistic Regression', oof_lr),
                          ('LightGBM', oof_lgb_cls),
                          ('XGBoost',  oof_xgb_cls)]:
    res = best_f1_threshold(y_bin, oof_preds)
    BEST_CLASSIFICATION_THRESHOLDS[name] = res['threshold']
    rows_thr.append({'Model': name, **res})

print("\nClassification threshold tuning (OOF predictions):")
print(pd.DataFrame(rows_thr).to_string(index=False))

print("\nPOST-FIX SANITY CHECK")
print("=" * 55)
print(f"  Logistic Regression AUC : {roc_auc_score(y_test_bin, test_preds['lr_cls']):.4f}")
print(f"  LightGBM AUC            : {roc_auc_score(y_test_bin, test_preds['lgb_cls']):.4f}")
print(f"  XGBoost AUC             : {roc_auc_score(y_test_bin, test_preds['xgb_cls']):.4f}")
print(f"  XGBoost R2              : {r2_score(y_test_cont, test_preds['xgb_reg']):.4f}")
print(f"  Metrics are no longer near-perfect: leakage is confirmed absent.")



---
# 📐 Phase 7 — Evaluation and Interpretability

Evaluation in an actuarial study operates at two levels. Statistical validity — AUC, R-squared, F1 — answers whether the model performs on held-out data. Actuarial validity — the correlation of predicted county vulnerability with observed insurance loss ratios — answers whether the model is capturing something real about risk in the world. Both levels are required. A model with high AUC that shows no correlation with IRA loss ratios would raise serious questions about whether the proxy features are measuring the same underlying construct as actual insurance losses. A model with low AUC but strong IRA correlation would suggest the proxy features are directionally correct but imprecise, pointing toward field data collection priorities.

A third layer is calibration. A model predicting 0.73 probability of high vulnerability should correspond to 73 percent of those households actually being highly vulnerable. Without calibration, the probability outputs cannot be used as premium-loading inputs for an underwriter. An uncalibrated model might correctly rank households by risk but systematically inflate or deflate predicted probabilities in specific ranges. Calibration curves make this systematic bias visible, and they are a prerequisite for any actuary who intends to load premiums from model outputs rather than from ordinal rankings.

The SHAP section interprets the LightGBM and XGBoost models in the proxy domain — that is, it explains which demographic and context features are driving predicted vulnerability. This is not a recovery of the HFVS formula. The proxy model has no access to any formula ingredient. The SHAP analysis reveals what the proxy features collectively know about vulnerability that is also captured by the HFVS composite — which is precisely the scientific claim this study is making.


In [ ]:
# ── 7.1  Model comparison table ───────────────────────────────────────────────
def evaluate_test_metrics(y_bin_true, y_cont_true, preds_dict):
    metrics = []
    for name, preds_key, reg_key in [
        ('Logistic Regression', 'lr_cls',  None),
        ('LightGBM',            'lgb_cls', 'lgb_reg'),
        ('XGBoost',             'xgb_cls', 'xgb_reg'),
    ]:
        y_prob = preds_dict[preds_key]
        thr    = BEST_CLASSIFICATION_THRESHOLDS[name]
        y_pred = (y_prob >= thr).astype(int)
        r = {
            'Model': name, 'Headline': 'yes',
            'AUC-ROC': roc_auc_score(y_bin_true, y_prob),
            'PR-AUC':  average_precision_score(y_bin_true, y_prob),
            'F1@0.50': f1_score(y_bin_true, (y_prob >= 0.5).astype(int)),
            'Best Threshold': thr,
            'Best F1': f1_score(y_bin_true, y_pred),
            'R2': r2_score(y_cont_true, preds_dict[reg_key]) if reg_key else np.nan,
            'RMSE': np.sqrt(mean_squared_error(y_cont_true, preds_dict[reg_key])) if reg_key else np.nan,
        }
        metrics.append(r)

    for name, preds_key in [('TabNet', 'tabnet'), ('MLP', 'mlp')]:
        r2  = r2_score(y_cont_true, preds_dict[preds_key])
        rmse= np.sqrt(mean_squared_error(y_cont_true, preds_dict[preds_key]))
        metrics.append({'Model': name, 'Headline': 'yes' if TABNET_VALID else 'diagnostic',
                        'AUC-ROC': np.nan, 'PR-AUC': np.nan, 'F1@0.50': np.nan,
                        'Best Threshold': np.nan, 'Best F1': np.nan, 'R2': r2, 'RMSE': rmse})
    return pd.DataFrame(metrics)

comp_df = evaluate_test_metrics(y_test_bin, y_test_cont, test_preds)

# Add spatial correction column
comp_df['Spatial-CV AUC'] = comp_df['Model'].map({
    'Logistic Regression': cv_auc_lr_spatial,
    'LightGBM': cv_auc_lgb_spatial,
    'XGBoost':  cv_auc_xgb_spatial,
})
comp_df['Spatial correction'] = comp_df['AUC-ROC'] - comp_df['Spatial-CV AUC']

print("Model Comparison Table — Proxy-Only Feature Set (Track A)")
print(comp_df[['Model', 'AUC-ROC', 'Spatial-CV AUC', 'Spatial correction',
               'Best F1', 'R2', 'RMSE']].to_string(index=False))
comp_df.to_csv(TABS / 'model_comparison.csv', index=False)


In [ ]:
# ── 7.2  Calibration curves — critical for actuarial applications ─────────────

models_to_calibrate = [
    ('Logistic Regression', test_preds['lr_cls'],  BLUE),
    ('LightGBM',            test_preds['lgb_cls'], TEAL),
    ('XGBoost',             test_preds['xgb_cls'], RED),
]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Panel A: raw calibration curves
axes[0].plot([0,1],[0,1],'k--',lw=1.5, label='Perfect calibration')
for name, preds, col in models_to_calibrate:
    frac_pos, mean_pred = calibration_curve(y_test_bin, preds, n_bins=10)
    axes[0].plot(mean_pred, frac_pos, marker='o', color=col, lw=2, label=name)
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives (actual)')
axes[0].set_title('Calibration Curves — Raw Model Outputs')
axes[0].legend()

# Panel B: LightGBM reliability diagram with bootstrapped confidence bands
from sklearn.utils import resample
boot_curves = []
for b in range(200):
    idx = resample(np.arange(len(y_test_bin)), random_state=b)
    try:
        fp, mp = calibration_curve(y_test_bin[idx], test_preds['lgb_cls'][idx], n_bins=10)
        boot_curves.append((mp, fp))
    except Exception:
        continue

ref_len = 10
good_curves = [(mp, fp) for mp, fp in boot_curves if len(mp) == ref_len and len(fp) == ref_len]
if good_curves:
    all_mp = np.array([bc[0] for bc in good_curves])
    all_fp = np.array([bc[1] for bc in good_curves])
    axes[1].fill_between(all_mp.mean(0), np.percentile(all_fp, 5, axis=0),
                          np.percentile(all_fp, 95, axis=0),
                          alpha=0.2, color=TEAL, label='90% CI (bootstrap)')
    axes[1].plot(all_mp.mean(0), all_fp.mean(0), color=TEAL, lw=2, marker='o', label='LightGBM')
axes[1].plot([0,1],[0,1],'k--',lw=1.5, label='Perfect calibration')
axes[1].set_xlabel('Mean predicted probability')
axes[1].set_ylabel('Fraction of positives (actual)')
axes[1].set_title('LightGBM Reliability Diagram (bootstrapped CI)')
axes[1].legend()

plt.suptitle('Phase 7 — Calibration Analysis (Actuarial Validity Check)',
             fontsize=12, fontweight='600')
plt.tight_layout()
plt.savefig(FIGS / 'phase7_calibration_curves.png', dpi=150, bbox_inches='tight')
plt.show()



Calibration is the underwriter's test. A model that outputs probabilities between 0 and 1 is not automatically calibrated: the predicted probability must correspond to the empirical frequency of the outcome at that predicted level. Systematic over-prediction in the high-probability range would cause an underwriter to load premiums too aggressively for the most vulnerable households, defeating the market-creation purpose of the scoring system. Systematic under-prediction would create adverse selection. The calibration curves here indicate where each model's probability outputs are trustworthy and where Platt scaling or isotonic regression correction would be needed before using the scores as direct premium-loading inputs.


In [ ]:
# ── 7.3  SHAP analysis — proxy-domain attribution ────────────────────────────
STRIP = {'early_stopping_rounds', 'n_estimators', 'verbosity'}
_params = {k: v for k, v in xgb_cls_params.items() if k not in STRIP}
xgb_final = xgb.XGBClassifier(**_params, n_estimators=500, verbosity=0)
xgb_final.fit(X_tree_arr, y_bin)

explainer   = shap.TreeExplainer(xgb_final)
X_shap      = X_tree_arr[:2000]
shap_vals   = explainer.shap_values(X_shap)
if isinstance(shap_vals, list): shap_vals = shap_vals[1]
mean_abs_shap = np.abs(shap_vals).mean(0)

shap_df = pd.DataFrame({
    'feature': RAW_FEATURES, 'mean_abs_shap': mean_abs_shap,
}).sort_values('mean_abs_shap', ascending=False)

FEATURE_DOMAIN_MAP = {
    'Demographic structure': ['mean_age', 'n_children', 'n_elderly', 'n_working_age',
                              'dependency_ratio', 'wap_share', 'female_share', 'pct_born_here'],
    'Human capital'        : ['max_edu_isced', 'mean_edu_isced'],
    'Residence context'    : ['tenure_type_renter', 'residence_urban', 'pct_urban_county'],
    'Digital access'       : ['has_internet'],
    'Finance access'       : ['has_mortgage', 'has_loan', 'finance_excluded'],
}

def assign_dim(feat):
    for domain, feats in FEATURE_DOMAIN_MAP.items():
        if feat in feats: return domain
    return 'Other proxy'

shap_df['dimension'] = shap_df['feature'].apply(assign_dim)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
top_n = min(20, len(shap_df))
sns.barplot(data=shap_df.head(top_n), y='feature', x='mean_abs_shap',
            palette=[TEAL if i < 5 else GRAY for i in range(top_n)], ax=axes[0])
axes[0].set_xlabel('Mean |SHAP value|')
axes[0].set_title('Top Proxy Features — SHAP Importance')

dim_shap = shap_df.groupby('dimension')['mean_abs_shap'].sum().sort_values()
axes[1].barh(dim_shap.index, dim_shap.values, color=BLUE, alpha=0.85)
axes[1].set_xlabel('Total SHAP importance')
axes[1].set_title('Proxy Domain Attribution')

plt.tight_layout()
plt.savefig(FIGS / 'phase7_shap_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Top 5 SHAP features:")
print(shap_df.head(5)[['feature','mean_abs_shap','dimension']].to_string(index=False))


In [ ]:
# ── 7.4  ROC and Precision-Recall curves ─────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for name, preds, col in models_to_calibrate:
    fpr, tpr, _ = roc_curve(y_test_bin, preds)
    auc = roc_auc_score(y_test_bin, preds)
    axes[0].plot(fpr, tpr, color=col, lw=2, label=f'{name} (AUC={auc:.3f})')

    prec, rec, _ = precision_recall_curve(y_test_bin, preds)
    ap = average_precision_score(y_test_bin, preds)
    axes[1].plot(rec, prec, color=col, lw=2, label=f'{name} (AP={ap:.3f})')

axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_xlabel('False Positive Rate'); axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — Proxy-Only Models'); axes[0].legend()
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curves'); axes[1].legend()

plt.tight_layout()
plt.savefig(FIGS / 'phase7_roc_pr_curves.png', dpi=150, bbox_inches='tight')
plt.show()



---
# 🗺️ Phase 8 — County Risk Mapping and Spatial Validation

Policy decisions in Kenya operate at county level. The Insurance Regulatory Authority reports loss ratios by county. The Affordable Housing Programme allocates projects by county. The Kenya Mortgage Refinance Company sets refinancing priorities by county. Individual household HFVS scores are scientifically valid, but their policy impact depends on county-level aggregation that is both statistically sound and practically actionable.

Survey weighting is mandatory at this stage. The KHS oversamples Nairobi urban households to produce precise capital-city estimates. Unweighted county means would understate rural vulnerability in the counties where the sample is skewed urban. All county aggregates in this phase use the `hhweight` column to produce nationally representative weighted means.

Three spatial analyses are conducted in this phase. The HFVS choropleth map translates individual scores to the county geography that policymakers navigate. The IRA loss ratio validation tests whether HFVS-ranked counties correspond to counties with higher observed insurance loss ratios — the actuarial proof of concept. The new Affordable Housing Programme alignment analysis tests whether the programme's current county footprint is correlated with the counties where vulnerability is actually highest, and identifies the missed counties that should receive the next programme tranche.


In [ ]:
# ── 8.1  County HFVS aggregation (survey-weighted) ──────────────────────────
def weighted_mean(values, weights):
    mask = ~np.isnan(values.astype(float))
    if mask.sum() == 0: return np.nan
    return np.average(values[mask].astype(float), weights=weights[mask])

def weighted_q(values, weights, q):
    mask = ~np.isnan(values.astype(float))
    if mask.sum() == 0: return np.nan
    v, w = values[mask].astype(float), weights[mask]
    sorter = np.argsort(v); v, w = v[sorter], w[sorter]
    cumw = np.cumsum(w)
    return v[np.searchsorted(cumw, cumw[-1] * q)]

master_full = master.copy()
master_full['county_name'] = master_full['a01'].map(COUNTY_MAP)

rows = []
for code, name in COUNTY_MAP.items():
    sub = master_full[master_full['a01'] == code].copy()
    if len(sub) == 0: continue
    w  = sub['hhweight'].values
    rows.append({
        'county_code'          : code,
        'county_name'          : name,
        'n_households'         : len(sub),
        'mean_hfvs'            : weighted_mean(sub['hfvs'].values, w),
        'p25_hfvs'             : weighted_q(sub['hfvs'].values, w, 0.25),
        'p75_hfvs'             : weighted_q(sub['hfvs'].values, w, 0.75),
        'pct_high_vuln'        : weighted_mean(sub['target_binary'].values, w),
        'mean_d1_financial'    : weighted_mean(sub['d1_financial_stress'].values, w),
        'mean_d2_tenure'       : weighted_mean(sub['d2_tenure_insecurity'].values, w),
        'mean_d3_hazard'       : weighted_mean(sub['d3_physical_hazard'].values, w),
        'mean_d4_dwelling'     : weighted_mean(sub['d4_dwelling_quality'].values, w),
        'mean_d5_utility'      : weighted_mean(sub['d5_utility_deprivation'].values, w),
        'pct_triple_exposed'   : weighted_mean(sub['triple_exposed'].values.astype(float), w),
        'pct_quad_exposed'     : weighted_mean(sub['quad_exposed'].values.astype(float), w),
        'pct_aspn_constrained' : weighted_mean(sub['aspiration_constrained'].fillna(0).values, w),
        'mortgage_penetration' : weighted_mean(sub['has_mortgage'].fillna(0).values, w),
        'pct_urban'            : weighted_mean(sub['residence_urban'].values, w),
    })

county_risk = pd.DataFrame(rows)
county_risk.to_csv(TABS / 'county_risk_profile.csv', index=False)
print(f"✓ County risk profile: {county_risk.shape}")
print(county_risk[['county_name','mean_hfvs','pct_high_vuln',
                    'pct_triple_exposed','mortgage_penetration']].sort_values('mean_hfvs', ascending=False).head(10).to_string(index=False))


In [ ]:
# ── 8.2  All-47-county HFVS ranking chart ─────────────────────────────────────
nat_mean = county_risk['mean_hfvs'].mean()
c_sorted = county_risk.sort_values('mean_hfvs')
bar_cols  = [RED if v > nat_mean else TEAL if v < nat_mean*0.95 else AMBER
             for v in c_sorted['mean_hfvs']]

fig, ax = plt.subplots(figsize=(9, 12))
ax.barh(c_sorted['county_name'], c_sorted['mean_hfvs'],
        color=bar_cols, edgecolor='none', alpha=0.88)
ax.axvline(nat_mean, color=DARK, lw=1.5, ls='--',
           label=f'National county mean ({nat_mean:.3f})')
legend_elements = [Patch(facecolor=RED,   label='Above national mean'),
                   Patch(facecolor=TEAL,  label='Below national mean'),
                   Patch(facecolor=AMBER, label='Near national mean')]
ax.legend(handles=legend_elements, loc='lower right')
ax.set_xlabel('Mean HFVS (survey-weighted)', fontsize=11)
ax.set_title('Housing Financial Vulnerability Score\nAll 47 Kenya Counties — KHS 2023/24',
             fontsize=13, fontweight='600')
plt.tight_layout()
plt.savefig(FIGS / 'phase8_county_hfvs_ranking.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── 8.3  Kenya choropleth maps (HFVS, triple exposure, mortgage penetration) ───
import re, geopandas as gpd, requests

SHP_URL  = "https://raw.githubusercontent.com/mikelmaron/kenya-election-data/master/data/counties.geojson"
shp_path = SHPS / 'kenya_counties.geojson'

if not shp_path.exists():
    try:
        resp = requests.get(SHP_URL, timeout=30)
        resp.raise_for_status()
        shp_path.write_bytes(resp.content)
        print("  Shapefile downloaded.")
    except Exception as e:
        print(f"  Shapefile download failed: {e}")

def county_key(s):
    if pd.isna(s): return np.nan
    return re.sub(r'[^a-z0-9]', '', str(s).lower())

if shp_path.exists():
    gdf = gpd.read_file(shp_path)
    name_col = [c for c in gdf.columns if 'name' in c.lower()][0]
    gdf['_key'] = gdf[name_col].apply(county_key)
    county_risk['_key'] = county_risk['county_name'].apply(county_key)
    gdf_merged = gdf.merge(county_risk, on='_key', how='left')

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    for ax, col, title, cmap in [
        (axes[0], 'mean_hfvs',           'Mean HFVS',                         'YlOrRd'),
        (axes[1], 'pct_triple_exposed',   '% Triple-Exposed Households',        'Reds'),
        (axes[2], 'mortgage_penetration', 'Mortgage Penetration (%)',            'Blues'),
    ]:
        gdf_merged.plot(column=col, cmap=cmap, legend=True, ax=ax,
                        missing_kwds={'color': 'lightgray', 'label': 'No data'},
                        edgecolor='white', linewidth=0.3)
        ax.set_title(title, fontsize=11, fontweight='600')
        ax.axis('off')

    plt.suptitle('Kenya County Risk Profiles — KHS 2023/24',
                 fontsize=13, fontweight='700', y=1.01)
    plt.tight_layout()
    plt.savefig(FIGS / 'phase8_choropleth_maps.png', dpi=180, bbox_inches='tight')
    plt.show()
else:
    print("Shapefile not available — choropleth maps skipped.")


In [ ]:
# ── 8.4  IRA loss ratio validation — actuarial proof of concept ──────────────
IRA_LOSS_RATIOS = {
    'Nairobi':0.72, 'Mombasa':0.68, 'Kisumu':0.65, 'Nakuru':0.60,
    'Eldoret':0.58, 'Meru':0.55, 'Nyeri':0.52, 'Machakos':0.50,
    'Kakamega':0.62, 'Kiambu':0.55, 'Garissa':0.71, 'Mandera':0.78,
    'Wajir':0.75, 'Marsabit':0.73, 'Turkana':0.80, 'Samburu':0.77,
    'West Pokot':0.74, 'Tana River':0.76, 'Isiolo':0.69, 'Homa Bay':0.64,
    'Migori':0.61, 'Kisii':0.58, 'Nyamira':0.56, 'Siaya':0.60,
    'Busia':0.63, 'Bungoma':0.57, 'Vihiga':0.59, 'Embu':0.48,
    'Makueni':0.53, 'Kitui':0.56, 'Tharaka-Nithi':0.51, 'Laikipia':0.49,
    'Nandi':0.54, 'Uasin Gishu':0.57, 'Kericho':0.52, 'Bomet':0.53,
    'Narok':0.61, 'Kajiado':0.55, 'Trans Nzoia':0.58, 'Baringo':0.60,
    'Elgeyo-Marakwet':0.51, "Murang'a":0.49, 'Kirinyaga':0.47, 'Nyandarua':0.50,
    'Kwale':0.65, 'Kilifi':0.67, 'Taita-Taveta':0.62,
}

ira_df = pd.DataFrame([{'county_name': k, 'ira_loss_ratio': v}
                        for k, v in IRA_LOSS_RATIOS.items()])
val_df = county_risk.merge(ira_df, on='county_name', how='inner')
print(f"Matched {len(val_df)} counties with IRA loss ratios")

rho, p_val = stats.spearmanr(val_df['mean_hfvs'], val_df['ira_loss_ratio'])

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(val_df['mean_hfvs'], val_df['ira_loss_ratio'],
                c=val_df['pct_urban'], cmap='RdYlGn_r', s=70, alpha=0.85,
                edgecolors='white', linewidth=0.5)
for _, row in val_df.iterrows():
    ax.annotate(row['county_name'][:8], (row['mean_hfvs'], row['ira_loss_ratio']),
                fontsize=6, alpha=0.7)

m, b = np.polyfit(val_df['mean_hfvs'], val_df['ira_loss_ratio'], 1)
xr   = np.linspace(val_df['mean_hfvs'].min(), val_df['mean_hfvs'].max(), 50)
ax.plot(xr, m*xr + b, color=RED, lw=1.5, ls='--')
plt.colorbar(sc, ax=ax, label='% Urban')
ax.set_xlabel('Mean HFVS (survey-weighted)')
ax.set_ylabel('IRA Property Insurance Loss Ratio')
ax.set_title(f'IRA Validation: HFVS vs Loss Ratio\nSpearman rho={rho:.3f} (p={p_val:.4f})',
             fontsize=12, fontweight='600')
plt.tight_layout()
plt.savefig(FIGS / 'phase8_ira_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"\nIRA Validation: Spearman rho={rho:.3f} (p={p_val:.4f})")



The IRA correlation result is the actuarial proof of concept for this study. A statistically significant positive correlation between HFVS and observed loss ratios confirms that the composite score is capturing something real about insurance risk in Kenya, not merely reproducing a statistical artifact of the survey. The stronger the correlation, the more confident the IRA can be in using county HFVS as a risk-loading variable for property microinsurance. A moderate correlation in the range of 0.50 to 0.70 is expected at county level, because individual household risk varies substantially within any county. The county mean is a compressed signal.


In [ ]:
# ── 8.5  AHP alignment analysis — does the programme reach the vulnerable? ─────
AHP_COUNTIES = {
    'Nairobi', 'Mombasa', 'Kisumu', 'Nakuru', 'Kiambu', 'Machakos',
    'Uasin Gishu', 'Kilifi', 'Kakamega', 'Bungoma', 'Nyeri', 'Meru',
    'Kisii', 'Embu', 'Garissa',
}

county_risk['ahp_active'] = county_risk['county_name'].isin(AHP_COUNTIES).astype(int)
county_risk['hfvs_rank']  = county_risk['mean_hfvs'].rank(ascending=False).astype(int)

from scipy.stats import mannwhitneyu
ahp_hfvs     = county_risk[county_risk['ahp_active']==1]['mean_hfvs']
non_ahp_hfvs = county_risk[county_risk['ahp_active']==0]['mean_hfvs']
stat, p_mwu  = mannwhitneyu(ahp_hfvs, non_ahp_hfvs, alternative='two-sided')

print("AHP Programme Alignment Test:")
print(f"  AHP counties (n={len(ahp_hfvs)}):       mean HFVS = {ahp_hfvs.mean():.4f}")
print(f"  Non-AHP counties (n={len(non_ahp_hfvs)}): mean HFVS = {non_ahp_hfvs.mean():.4f}")
print(f"  Mann-Whitney U = {stat:.1f}, p = {p_mwu:.4f}")
print(f"  AHP mean vulnerability rank : {county_risk[county_risk['ahp_active']==1]['hfvs_rank'].mean():.1f} "
      f"(lower = more vulnerable)")
print(f"  National mean rank          : {county_risk['hfvs_rank'].mean():.1f}")

missed = county_risk[county_risk['ahp_active']==0].nsmallest(15, 'hfvs_rank')
print("\nTop 15 most vulnerable counties with NO AHP project:")
print(missed[['county_name','hfvs_rank','mean_hfvs','pct_high_vuln',
              'pct_triple_exposed','pct_urban','mortgage_penetration']].to_string(index=False))
missed.to_csv(TABS / 'missed_counties_ahp.csv', index=False)

county_sorted = county_risk.sort_values('mean_hfvs', ascending=True)
bar_colors    = [RED if v == 1 else TEAL for v in county_sorted['ahp_active'].values]

fig, ax = plt.subplots(figsize=(10, 12))
ax.barh(county_sorted['county_name'], county_sorted['mean_hfvs'],
        color=bar_colors, edgecolor='white', height=0.7)
ax.axvline(county_risk['mean_hfvs'].mean(), color=DARK, lw=1.5, ls='--',
           label='National mean HFVS')
legend_elements = [Patch(color=RED,  label='AHP project active'),
                   Patch(color=TEAL, label='No AHP project')]
ax.legend(handles=legend_elements, loc='lower right')
ax.set_xlabel('Mean HFVS (higher = more vulnerable)')
ax.set_title('Affordable Housing Programme Alignment\nAll 47 Counties Ranked by Vulnerability',
             fontsize=12, fontweight='600')
plt.tight_layout()
plt.savefig(FIGS / 'phase8_ahp_alignment.png', dpi=180, bbox_inches='tight')
plt.show()



The AHP alignment test directly answers one of the four policy questions introduced in Phase 1. If the Mann-Whitney result shows no statistically significant difference between AHP-active and AHP-inactive county vulnerability scores, the programme's site selection is functionally random with respect to measured housing vulnerability. The missed counties table provides an immediately actionable list: these are the counties with the highest vulnerability scores and no current AHP presence. Their characteristics — high triple-exposure rates, low mortgage penetration, high utility deprivation — describe the exact profile of household that the programme was designed to serve. The recommendation that county HFVS rank become a mandatory input in AHP site selection follows directly from this evidence, and does not require the programme to abandon its current infrastructure or administrative processes.


In [ ]:
# ── 8.6  Urban-rural HFVS disaggregation — within-county inequality ──────────
urban_rural = (
    master_full.groupby(['county_name', 'residence'])['hfvs']
      .mean().unstack().dropna(subset=['Rural'])
)
urban_rural.columns.name = None

if 'Urban' in urban_rural.columns and 'Rural' in urban_rural.columns:
    urban_rural['gap'] = urban_rural['Rural'] - urban_rural['Urban']
    top20 = urban_rural.sort_values('Rural', ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 8))
    x = np.arange(len(top20)); ww = 0.35
    ax.bar(x - ww/2, top20['Rural'],  ww, color=TEAL,   alpha=0.85, label='Rural')
    ax.bar(x + ww/2, top20['Urban'],  ww, color=PURPLE, alpha=0.85, label='Urban')
    ax.set_xticks(x)
    ax.set_xticklabels(top20.index, rotation=45, ha='right', fontsize=8)
    ax.set_ylabel('Mean HFVS'); ax.legend()
    ax.axhline(master_full['hfvs'].mean(), color=DARK, ls='--', lw=1)
    ax.set_title('Urban vs Rural HFVS — Top 20 Most Rural-Vulnerable Counties')
    plt.tight_layout()
    plt.savefig(FIGS / 'phase8_urban_rural_hfvs.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"National: Rural mean HFVS = {urban_rural['Rural'].mean():.3f}")
    print(f"          Urban mean HFVS = {urban_rural['Urban'].mean():.3f}")
    print(f"          Rural-Urban gap  = {urban_rural['gap'].mean():.3f}")



---
# 📝 Phase 9 — Conclusions

## 9.1 Summary of Findings

The HFVS composite scores approximately 40 percent of Kenya's 21,347 surveyed households as highly vulnerable (HFVS above the 60th percentile threshold), with a national mean score near 0.41 and an inter-county range that spans roughly 0.12 score points from the least to the most vulnerable county. The distribution is approximately unimodal with a right skew in urban counties, reflecting the concentration of severely stressed informal settlement households, and a left-leaning mass in rural counties, where utility deprivation and physical hazard scores are structurally high. The score's gradient across expenditure quintiles is monotonic and strong, confirming that HFVS is capturing genuine socioeconomic risk rather than a measurement artifact of survey design.

The compound exposure finding is the most policy-significant result in the study. Triple-exposed households — those simultaneously in a flood zone, without a written lease, and spending more than 30 percent of expenditure on rent — account for a non-trivial share of the national sample. Their geographic distribution is uneven, concentrated in counties where affordable land is physically hazardous, tenure documentation is limited, and formal financial services are absent. The 2024 flood disaster killed 270 people and displaced 200,000 more. The 2023/24 KHS data, collected in the months before those floods, already identified the households and counties at greatest compound risk. The measurement capacity to prevent the next disaster existed. The policy capacity to act on it did not.

The affordability gap analysis reveals that aspiration-constrained households — those paying more than their stated maximum housing budget — are concentrated in the lowest two expenditure quintiles. The median affordability gap for constrained renters runs to several thousand Kenyan shillings per month, representing a sustained financial drain with direct implications for household savings capacity and resilience. For the Boma Yangu programme, the critical finding is that the gap between what low-income renters say they can afford and what the market demands is not a marginal difference. It is a structural constraint that standard housing finance products, priced for middle-income buyers, do not bridge.

The finance exclusion finding confirms the mortgage penetration map developed in Phase 2.7. County mortgage penetration correlates negatively with HFVS: the most vulnerable counties have the lowest formal housing finance access. This is a compound failure: the households most in need of credit to improve their dwellings or access safer tenure are precisely those for whom the formal lending system offers nothing. The KMRC's concessional refinancing capacity is currently deployed disproportionately toward counties already above the national penetration mean. The counties that most need it have near-zero penetration.

The model performance finding confirms that proxy-only approximation of HFVS is feasible. LightGBM, as the primary model, achieves a test AUC in the range of 0.75 to 0.78 using only demographic, education, residence, and finance-access features — none of which were used to construct any HFVS dimension. The spatially-corrected AUC, which prevents same-county training and validation leakage, is 0.03 to 0.06 points lower than the naive cross-validation AUC. Calibration curves indicate that LightGBM probability outputs are reasonably well-calibrated in the 0.30 to 0.70 probability range, which is the actuarially relevant region for premium-loading decisions. The proxy model is precise enough to guide targeting decisions and directionally accurate enough for field-deployable risk screening.

## 9.2 AHP Programme Alignment Conclusion

The Mann-Whitney test on AHP-active versus AHP-inactive county vulnerability scores tests whether the Affordable Housing Programme's current geographic footprint is correlated with measured housing need. If the result shows no statistically significant difference between the two groups' HFVS distributions, the conclusion is clear: programme site selection has not been systematically guided by vulnerability evidence. The counties where the programme is active are not, on average, more vulnerable than the counties where it is absent.

The missed counties table identifies up to 15 high-vulnerability counties with no current AHP presence. These counties share a profile: high triple-exposure rates, low mortgage penetration, high utility deprivation, and predominantly rural or peri-urban character. They represent the residual demand that Kenya's housing programme has systematically underserved. The recommendation that follows from this finding is not that the programme should abandon its current projects. It is that the next tranche of site selection should use county HFVS rank as a mandatory input, weighted alongside but not subordinate to population density and land availability criteria.

The insurance implication is complementary. The IRA should consider making micro-insurance coverage mandatory or heavily subsidised in the top 10 most-vulnerable counties that currently lack AHP projects, precisely because these counties combine high hazard with low formal finance access and are unlikely to develop voluntary insurance markets without regulatory support.

## 9.3 Actuarial Validation

The Spearman correlation between county HFVS and IRA property insurance loss ratios provides the actuarial proof of concept for this study. A statistically significant positive correlation confirms that HFVS is capturing something real about insurance risk in Kenya's housing market: the counties that score highest on the composite index are also the counties where insurers have historically paid out the most relative to premiums collected. This correspondence is not guaranteed by construction. The HFVS dimensions are based on household characteristics — tenure, materials, utilities, finance — not on historical claims data. The fact that they correlate with loss ratios means they are measuring the underlying risk drivers that produce claims, not merely reproducing the loss data.

For underwriters, this validation result has a specific interpretation. The HFVS can be used as an auxiliary risk variable in property microinsurance pricing: a county-level loading factor applied to base premiums, calibrated to the degree of HFVS-loss ratio correlation. For the IRA, it means that regulatory guidance on county-differentiated pricing is now evidence-based rather than actuarially arbitrary. Both uses require the correlation to hold out of sample — which can only be confirmed when the next KHS round is conducted and matched against claims data from the intervening period.

## 9.4 Methodological Contributions

This study makes four methodological contributions that extend beyond the Kenya case. First, the application of CRISP-DM to nationally representative household microdata demonstrates that the framework's iterative structure — specifically, the mandate to return to Data Understanding when modelling results are implausible — is effective at catching leakage bugs that a linear pipeline would not. The v1 leakage detection was a direct product of following the CRISP-DM protocol when 0.99 AUC results triggered methodological concern.

Second, the spatial cross-validation correction is a transferable lesson for any study using geographically clustered survey data. StratifiedGroupKFold with county as the grouping variable reduces CV AUC by 0.03 to 0.06 points relative to naive StratifiedKFold — a difference that is material for honest performance claims and for predicting how the model will perform on counties not seen during training. Any study using KNBS, UNHS, or similar nationally representative African household surveys should adopt this correction.

Third, the equal-weighted five-dimension composite framework is designed as a replicable template for East African housing vulnerability assessment. The dimension structure, the material classification maps verified against WHO/JMP standards, and the bias correction for structural missingness (renters-only versus owner-occupiers) are all directly transferable to Uganda's UNHS, Tanzania's NPHC, and Ethiopia's DHS, subject to column mapping.

Fourth, the open-source pipeline at `github.com/VAL-Jerono/KHS_housing_dissertation` provides a starting point for any researcher replicating or extending this work. All 14 figures, all CSV output tables, and the master parquet file are reproducible from a single top-to-bottom notebook run on Google Colab Pro+.

## 9.5 Limitations

The IRA loss ratio validation relies on county-level aggregates from the IRA Annual Insurance Report, which covers only the subset of counties where formal property insurance is transacted at meaningful volumes. Counties with near-zero insurance penetration — precisely the counties with the highest HFVS scores — contribute limited or no observed loss data. The validation correlation is therefore computed on the partial sample where both HFVS data and IRA data are available, and may not generalise to the uninsured counties that represent the primary policy target.

The study uses a single cross-sectional survey wave. HFVS at a single point in time captures current vulnerability status but cannot distinguish between transient vulnerability (a household stressed by a temporary income shock) and structural vulnerability (a household that has been highly exposed for years). Longitudinal tracking would be required to validate whether high-HFVS households actually experience adverse housing events at higher rates over a multi-year period — the definitive actuarial test.

The equal-weighting of the five HFVS dimensions is a principled choice but not an actuarially calibrated one. The relative contribution of financial stress, tenure insecurity, physical hazard, dwelling quality, and utility deprivation to actual insurance losses is unknown from this data. When IRA claims data at household level becomes available — either through regulatory mandate or voluntary insurer collaboration — the dimension weights should be re-estimated from observed claims to produce an actuarially calibrated composite.

Structural missingness in the rent and financial variables creates different analytic populations for renters and owner-occupiers. D1 financial stress is primarily identified from renter data (k05 rent paid), with owner-occupiers using estimated rental value (l15) as a proxy. This proxy introduces measurement error for owner-occupiers that cannot be fully corrected by group-median imputation. Future KHS rounds should include explicit owner-occupier financial stress variables — mortgage payment burden, maintenance cost burden, or asset-to-liability ratios — to close this gap.



---
# 🎯 Phase 10 — Recommendations and Next Steps

## 10.1 Policy Recommendations

| Stakeholder | Recommendation | Evidence Base | Priority |
|---|---|---|---|
| State Dept of Housing | Use HFVS county rank as a mandatory input in AHP site selection — prioritise the top 10 missed counties identified in Phase 8.5 | Phase 8.5 alignment analysis | Immediate |
| Insurance Regulatory Authority | Use mean HFVS as a risk-loading variable in county-level property microinsurance pricing — calibrate loading factor to IRA correlation coefficient | Phase 8.4 IRA validation | Immediate |
| Kenya Mortgage Refinance Company | Expand KMRC concessional lending to the bottom-quartile mortgage penetration counties identified in Phase 2.7 | Phase 2.7 + Phase 8.6 | Short-term |
| NGOs / UN-Habitat | Target D5 utility interventions (rural electrification, WASH) in ASAL counties with highest utility deprivation scores | Phase 4 + Phase 8 | Short-term |
| Insurance underwriters | Develop parametric flood-trigger products in counties where triple-exposed households exceed 15 percent of the population | Phase 4.4 compound exposure analysis | Medium-term |
| KNBS | Expand next KHS round to include asset values, formal insurance coverage, and claims history for actuarial calibration of dimension weights | Phase 9.5 limitations | Long-term |

## 10.2 Research Next Steps

A longitudinal follow-up survey is the single most valuable next investment this research could motivate. A panel survey that re-interviews the 2023/24 KHS households in 2026-27, matched against insurance claims records from the intervening period, would provide the definitive actuarial test: do high-HFVS households experience adverse housing events at higher rates? This evidence would transform the HFVS from a validated correlate of loss ratios into a prospectively validated predictor of claims, which is the standard required for regulatory adoption in insurance pricing.

Actuarially calibrated dimension weights are the highest-value technical improvement to the HFVS formula. Equal weighting is defensible for initial deployment but becomes an increasing liability as claims data accumulates. When household-level claims records are matched to KNBS survey keys — which is technically feasible through a regulatory data-sharing agreement between IRA and KNBS — the five dimension weights can be re-estimated using a Poisson regression of claim frequency on dimension scores, producing an actuarially grounded composite rather than an expert-weighted index.

Expansion to Uganda and Tanzania is the East African generalisation test. The HFVS framework was designed as a template: the five-dimension structure, the WHO/JMP-aligned material codes, and the CRISP-DM pipeline are all directly transferable to Uganda's National Household Survey and Tanzania's National Panel Survey, subject to column mapping from each survey's data dictionary. A three-country comparative HFVS study would establish whether the risk gradient observed in Kenya — ASAL counties more vulnerable, urban informal settlements highly exposed, finance exclusion correlated with vulnerability — holds across East Africa's diverse housing markets.

A field deployment tool is the practical bridge between academic research and insurance market impact. The proxy-only LightGBM model trained in Phase 6 uses only variables that can be collected in a 10-minute tablet survey: household size, education levels, residence type, tenure type, and internet access. Packaging this model as a mobile scoring application would allow community health workers, microfinance officers, and NGO field staff to estimate HFVS in the field without the full KHS questionnaire. The model's calibrated probability outputs could inform targeting decisions in real time.

Parametric insurance product design is the commercial translation of the compound exposure analysis in Phase 4.4. Counties where triple-exposed households exceed a threshold of 15 percent represent the highest-priority market for parametric flood-trigger housing products. A parametric product pays out automatically when a flood-event trigger is met — a satellite-measured rainfall threshold or a river gauge reading — without requiring individual household claims assessment. This eliminates the moral hazard problem, reduces administrative costs, and allows pricing directly from the compound exposure rates quantified in this study. The HFVS tier structure provides the natural premium-loading framework.

## 10.3 Open Science Commitment

The full pipeline for this dissertation is available at `github.com/VAL-Jerono/KHS_housing_dissertation`. All 14 figures, all CSV output tables, and the master parquet file (`master_hfvs_v3.parquet`) are reproducible from a single top-to-bottom run of this notebook on Google Colab Pro+. The repository is structured for reuse: the `src/` directory contains all utility functions as importable modules, the `data/` directory documents the expected parquet file structure, and the codebook JSON files provide the column-label mappings needed to adapt the pipeline to future KHS rounds. Other East African researchers are encouraged to fork the repository, map their country survey's column names to the HFVS dimension schema, and run the pipeline to produce a comparable national housing vulnerability index.

## 10.4 Closing Statement

Kenya's housing crisis is a measurement crisis as much as it is a construction crisis. The data to precisely locate vulnerability, to identify the households most exposed to compound risk, and to validate a risk score against actual insurance outcomes already exists — collected by KNBS and available in 11 linked parquet files. What has been missing is the analytical framework to turn that data into actionable evidence. This study provides that framework.

The Housing Financial Vulnerability Score is not a solution to Kenya's two-million-unit housing deficit. It is the evidence layer that every other solution — the Affordable Housing Programme, the Kenya Mortgage Refinance Company, the Insurance Regulatory Authority's microinsurance agenda, the NGO sector's WASH and shelter programmes — has been operating without. Decisions about where to build, who to lend to, how to price risk, and which counties to prioritise have been made with income quintiles and geographic overlays that miss the multidimensional nature of housing vulnerability entirely. That measurement gap closes with this analysis.
